# Home Credit Default Risk — Feature Engineering

This notebook builds on the findings from the EDA (01_eda): it cleans the
identified data quality issues, encodes categorical variables, and prepares
the dataset for modeling. Where relevant, it also incorporates the linked
tables (bureau, previous applications, payment history) via aggregation.

## Key Findings

* Dataset scope: started from `application_train.csv` (307,511 rows, 122 columns) and ended with 244 columns — 122 original features cleaned/encoded, plus engineered features from all six linked tables except `bureau_balance` (deliberately excluded).
* Data quality fixes: `DAYS_EMPLOYED`'s 365243 placeholder (55,374 rows) replaced with NaN and flagged; `AMT_INCOME_TOTAL` capped at the 99th percentile (472,500), affecting 3,014 applicants including the 117M data-entry error found in the EDA; rare placeholder values in `CODE_GENDER` ("XNA", 4 rows) and `NAME_FAMILY_STATUS` ("Unknown", 2 rows) converted to NaN.
* Categorical encoding: one-hot encoding of 16 categorical columns expanded the dataset from 122 to 245 columns — driven mainly by the two high-cardinality columns, `ORGANIZATION_TYPE` and `OCCUPATION_TYPE`.
* Redundant building/apartment features removed: the `_AVG`, `_MODE`, and `_MEDI` variants of all 14 building features (identified among the EDA's 41 columns with 50-70% missingness) correlate at 0.966-0.989 with each other — kept only `_AVG`, dropping 28 columns, then median-imputed the remaining 2,516,080 missing values across those 14 features (no separate flags, given their weak correlation with `TARGET`).
* Structural vs. random missingness distinguished explicitly: `OWN_CAR_AGE`'s missing values align almost perfectly (202,924 of 202,929) with applicants who don't own a car, so filled with 0 using the existing `FLAG_OWN_CAR_N` rather than a new flag; `EXT_SOURCE_1/2/3` instead got median imputation plus separate `_MISSING` flags, since their absence may itself carry predictive signal.
* Six linked tables considered; `bureau_balance` deliberately excluded — it would need an extra join via `SK_ID_BUREAU`, its signal largely overlaps with overdue-related columns already aggregated from `bureau`, and its size didn't justify the added complexity. The remaining five (`bureau`, `previous_application`, `POS_CASH_balance`, `credit_card_balance`, `installments_payments`) were aggregated to one row per applicant and merged in — `bureau` with a thorough, from-scratch approach, the other four with a lighter, more standardized one. Missing values from these merges were filled with 0, since absence here means no recorded activity, not an unknown quantity.
* Coverage of the linked tables ranged from 28.3% (`credit_card_balance`) to 94.8% (`installments_payments`) — a key lesson was that these tables span both the train and test populations combined, so true coverage had to be verified directly against `app_train` rather than inferred from unique-ID counts.
* Four domain-informed custom ratio features added at the aggregate level (sum/sum, not per-row): `BUREAU_DEBT_CREDIT_RATIO`, `PREV_CREDIT_APPLICATION_RATIO`, `CC_UTILIZATION_RATIO`, and `INST_PAYMENT_RATIO` (plus `INST_DAYS_LATE_MEAN`, a direct payment-punctuality measure). Each required explicit handling of zero-denominator cases, since pandas silently produces `inf`/`NaN` rather than raising an error.
* A final sweep across the fully merged dataset caught 16 lower-missingness columns that had been missed earlier (457,296 values total) — six credit bureau inquiry columns, `DAYS_EMPLOYED` itself, `TOTALAREA_MODE`, and eight small residual columns — all resolved. `app_train` now has 244 columns and zero missing values.

Let's first import the libraries and set up the same plot style as in the EDA notebook, for consistency.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', 200)

plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
plt.rcParams['axes.titleweight'] = 'bold'
plt.rcParams['axes.titlesize'] = 13
plt.rcParams['figure.figsize'] = (7, 4)

Let's load the same training data we explored in the EDA notebook, so we can start applying the cleaning and encoding steps identified there.

In [2]:
app_train = pd.read_csv('/kaggle/input/competitions/home-credit-default-risk/application_train.csv')

print('Shape:', app_train.shape)

Shape: (307511, 122)


## Cleaning DAYS_EMPLOYED

In the EDA, we found that DAYS_EMPLOYED contains a placeholder value of
365243 for 55,374 applicants (mostly pensioners), which distorts the
feature's statistics. We'll replace this placeholder with NaN so the
numerical values reflect only real employment durations.

However, simply replacing it with NaN would discard the information that
these applicants are not currently employed — which may itself be
predictive of credit risk. To preserve this signal, we add a separate
boolean flag column (`DAYS_EMPLOYED_ANOM`) before doing the replacement,
so the "not employed" information survives independently of the cleaned
numerical feature.

In [3]:
app_train['DAYS_EMPLOYED_ANOM'] = app_train['DAYS_EMPLOYED'] == 365243
app_train['DAYS_EMPLOYED'] = app_train['DAYS_EMPLOYED'].replace(365243, np.nan)

print('Anomalous entries flagged:', app_train['DAYS_EMPLOYED_ANOM'].sum())
print('Missing values in DAYS_EMPLOYED now:', app_train['DAYS_EMPLOYED'].isnull().sum())

Anomalous entries flagged: 55374
Missing values in DAYS_EMPLOYED now: 55374


Confirmed: 55,374 entries flagged and cleaned — matching the anomaly count
found in the EDA. `DAYS_EMPLOYED` now contains only real employment
durations, while `DAYS_EMPLOYED_ANOM` preserves the "not currently
employed" signal as a separate feature.

## Handling the AMT_INCOME_TOTAL Outlier

In the EDA, we found a single extreme outlier in AMT_INCOME_TOTAL
(117,000,000 — more than 6x the second-highest value), traced to a likely
data entry error, within a broader set of ~250 applicants reporting income
above 1,000,000.

There are two common ways to handle this: removing the affected row(s), or
capping the extreme value(s) at a chosen threshold. Removing a row discards
not just the suspicious value but the applicant's entire record, including
their TARGET label — a real cost given the limited size and class
imbalance of this dataset (only 8.1% TARGET=1, and the 117M row happens to
be one of these minority cases). It would also require deciding, row by
row, which of the ~250 high-income applicants are genuine data errors
versus real (if rare) high earners — a judgment we can't reliably make
from the data alone.

Capping (winsorizing) avoids this problem: it keeps every row intact and
only limits how extreme a value is allowed to be, regardless of whether it
reflects a data error or a legitimate rare case. We'll cap
AMT_INCOME_TOTAL at the 99th percentile: any value above this threshold is
replaced with the threshold value itself, while all other values stay
unchanged.

In [4]:
income_cap = app_train['AMT_INCOME_TOTAL'].quantile(0.99)
print('99th percentile (cap threshold):', income_cap)

print('Applicants affected by capping:', (app_train['AMT_INCOME_TOTAL'] > income_cap).sum())

app_train['AMT_INCOME_TOTAL'] = app_train['AMT_INCOME_TOTAL'].clip(upper=income_cap)

print('New max AMT_INCOME_TOTAL:', app_train['AMT_INCOME_TOTAL'].max())

99th percentile (cap threshold): 472500.0
Applicants affected by capping: 3014
New max AMT_INCOME_TOTAL: 472500.0


472,500 as the cap threshold looks reasonable — about 2.3x the 75th
percentile (202,500) found in the EDA, consistent with the heavily
right-skewed shape of this feature. 3,014 applicants (~1%) were affected,
including the 117M data-entry error identified earlier. AMT_INCOME_TOTAL
is now bounded to a more plausible range without discarding any rows.

## Encoding Categorical Features

Before encoding, we clean up two rare placeholder values identified in the
EDA: CODE_GENDER's 'XNA' (4 rows) and NAME_FAMILY_STATUS's 'Unknown' (2
rows). Both were found to be data placeholders rather than meaningful
categories, given how rarely they occur. We convert them to NaN so the
encoding step treats them as missing values instead of as their own
category.

Note: ORGANIZATION_TYPE's 'XNA' is different — it's a large, meaningful
group (55,374 rows) representing applicants without a current employer, so
we leave it as its own category there.

In [5]:
app_train['CODE_GENDER'] = app_train['CODE_GENDER'].replace('XNA', np.nan)
app_train['NAME_FAMILY_STATUS'] = app_train['NAME_FAMILY_STATUS'].replace('Unknown', np.nan)

print('CODE_GENDER missing:', app_train['CODE_GENDER'].isnull().sum())
print('NAME_FAMILY_STATUS missing:', app_train['NAME_FAMILY_STATUS'].isnull().sum())

CODE_GENDER missing: 4
NAME_FAMILY_STATUS missing: 2


Confirmed: CODE_GENDER now has 4 missing values and NAME_FAMILY_STATUS has
2 — exactly matching the placeholder counts found in the EDA. Both
placeholders are now properly represented as missing values instead of
misleading categories, ready for the encoding step.

Recall from the EDA that 16 columns have an `object` dtype (categorical/
text) — from binary flags like FLAG_OWN_CAR to high-cardinality features
like ORGANIZATION_TYPE (58 categories). Since models need numeric input,
we need to encode these before modeling.

We use one-hot encoding, which creates one binary (0/1) column per
category. The main downside: high-cardinality columns like
ORGANIZATION_TYPE and OCCUPATION_TYPE alone add ~76 new columns.
Alternatives exist — label encoding, frequency encoding, target/mean
encoding, or LightGBM's native categorical support — but one-hot is the
only one that works consistently for both our planned baseline (Logistic
Regression, which cannot handle native categoricals or false ordinal
relationships) and our main model (LightGBM), without the data leakage
risk that target encoding would introduce. We may revisit this trade-off
later if dimensionality becomes a practical problem.

In [6]:
categorical_cols = app_train.select_dtypes('object').columns.tolist()
print('Categorical columns to encode:', len(categorical_cols))

app_train = pd.get_dummies(app_train, columns=categorical_cols)

print('Shape after encoding:', app_train.shape)

Categorical columns to encode: 16
Shape after encoding: (307511, 245)


Shape grew from 123 to 245 columns — exactly as expected. This can be
traced precisely: the EDA found 140 total categories across the 16
categorical columns (sum of each column's unique value count). After
cleaning, CODE_GENDER dropped from 3 to 2 categories and
NAME_FAMILY_STATUS from 6 to 5 (the placeholder values are now NaN
instead of a category), leaving 138 actual dummy columns. So:
123 columns (122 + DAYS_EMPLOYED_ANOM) − 16 original categorical columns +
138 new dummy columns = 245.

This confirms the trade-off discussed above: one-hot encoding roughly
doubled the number of features, driven mainly by the two high-cardinality
columns (ORGANIZATION_TYPE, OCCUPATION_TYPE). All categorical columns are
now represented as binary (0/1) features, ready for modeling.

quick check:

In [7]:
app_train.dtypes.value_counts()

bool       139
float64     66
int64       40
Name: count, dtype: int64

confirmed: type "object" is no longer among the counted dtype values.

The numbers also check out precisely: bool (139) = the 138 one-hot dummy
columns + the DAYS_EMPLOYED_ANOM flag added earlier. float64 increased
from 65 (in the EDA) to 66, while int64 dropped from 41 to 40 — because
replacing the DAYS_EMPLOYED placeholder with NaN forced pandas to upcast
that column from int64 to float64 (NaN can't be stored in an integer
column). In total: 139 + 66 + 40 = 245, matching the shape confirmed
above.

## Imputing EXT_SOURCE_1/2/3

Recall from the EDA: EXT_SOURCE_1/2/3 are by far our strongest predictors,
but with very different missingness — EXT_SOURCE_1 at 56%, EXT_SOURCE_3 at
20%, and EXT_SOURCE_2 at just 0.2%.

We impute missing values with the median (more robust to skew than the
mean) so all three columns have complete data for modeling. However,
since these are our strongest predictors, simply filling in a "neutral"
value could hide something meaningful — the absence of an external score
(e.g. no credit bureau record available) might itself be associated with
risk. To preserve this information, we add a separate missing-indicator
flag column for each (e.g. `EXT_SOURCE_1_MISSING`), following the same
approach we used for `DAYS_EMPLOYED_ANOM` earlier.

In [8]:
ext_source_cols = ['EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3']

for col in ext_source_cols:
    app_train[f'{col}_MISSING'] = app_train[col].isnull()
    app_train[col] = app_train[col].fillna(app_train[col].median())

print('Remaining missing values:')
print(app_train[ext_source_cols].isnull().sum())

print('\nFlagged as originally missing:')
print(app_train[[f'{col}_MISSING' for col in ext_source_cols]].sum())

Remaining missing values:
EXT_SOURCE_1    0
EXT_SOURCE_2    0
EXT_SOURCE_3    0
dtype: int64

Flagged as originally missing:
EXT_SOURCE_1_MISSING    173378
EXT_SOURCE_2_MISSING       660
EXT_SOURCE_3_MISSING     60965
dtype: int64


All three EXT_SOURCE columns are now complete, with missingness preserved
separately in the three flag columns — matching the EDA's missing-value
percentages exactly. The strongest predictors in our dataset are now
ready for modeling without losing the information about which values were
originally missing.

quick check:

In [9]:
for col in ext_source_cols:
    missing_count = app_train[f'{col}_MISSING'].sum()
    missing_pct = missing_count / len(app_train) * 100
    print(f'{col}: {missing_count} missing ({missing_pct:.6f}%)')

EXT_SOURCE_1: 173378 missing (56.381073%)
EXT_SOURCE_2: 660 missing (0.214626%)
EXT_SOURCE_3: 60965 missing (19.825307%)


Confirmed: recomputing the missing percentages from the flag columns
(56.381073%, 0.214626%, 19.825307%) matches the EDA's original values
exactly. The missing-indicator flags accurately preserve the original
missingness pattern for all three EXT_SOURCE columns.

## Simplifying the Building/Apartment Features

In the EDA, we found 41 columns with 50-70% missingness. Before assuming
what these columns are based on our EDA notes, let's list them explicitly
and verify.

In [10]:
app_train_raw = pd.read_csv('/kaggle/input/competitions/home-credit-default-risk/application_train.csv')

missing_pct = app_train_raw.isnull().sum() / len(app_train_raw) * 100
cols_50_70 = missing_pct[(missing_pct >= 50) & (missing_pct < 70)].sort_values(ascending=False)

print('Number of columns with 50-70% missingness:', len(cols_50_70))
print(cols_50_70)

Number of columns with 50-70% missingness: 41
COMMONAREA_AVG              69.872297
COMMONAREA_MEDI             69.872297
COMMONAREA_MODE             69.872297
NONLIVINGAPARTMENTS_AVG     69.432963
NONLIVINGAPARTMENTS_MODE    69.432963
NONLIVINGAPARTMENTS_MEDI    69.432963
FONDKAPREMONT_MODE          68.386172
LIVINGAPARTMENTS_AVG        68.354953
LIVINGAPARTMENTS_MODE       68.354953
LIVINGAPARTMENTS_MEDI       68.354953
FLOORSMIN_MODE              67.848630
FLOORSMIN_MEDI              67.848630
FLOORSMIN_AVG               67.848630
YEARS_BUILD_MODE            66.497784
YEARS_BUILD_MEDI            66.497784
YEARS_BUILD_AVG             66.497784
OWN_CAR_AGE                 65.990810
LANDAREA_MODE               59.376738
LANDAREA_MEDI               59.376738
LANDAREA_AVG                59.376738
BASEMENTAREA_MODE           58.515956
BASEMENTAREA_AVG            58.515956
BASEMENTAREA_MEDI           58.515956
EXT_SOURCE_1                56.381073
NONLIVINGAREA_MEDI          55.179164
NONL

Breaking this list down: 36 of the 41 columns are indeed our known
pattern — 12 building/apartment features (COMMONAREA, NONLIVINGAPARTMENTS,
LIVINGAPARTMENTS, FLOORSMIN, YEARS_BUILD, LANDAREA, BASEMENTAREA,
NONLIVINGAREA, ELEVATORS, APARTMENTS, ENTRANCES, LIVINGAREA), each in
three variants (AVG/MODE/MEDI). This confirms our approach for these.

The remaining 5 columns are not part of that pattern:
- `FONDKAPREMONT_MODE`, `WALLSMATERIAL_MODE`, `HOUSETYPE_MODE` — categorical
  building-related columns with no AVG/MEDI counterpart. These were already one-hot encoded earlier, so their
  missingness is already handled implicitly.
- `EXT_SOURCE_1` — already addressed in the previous step.
- `OWN_CAR_AGE` — not a building feature at all, but the age of the
  applicant's car. This hasn't been addressed yet and needs its own
  decision.

Worth noting: two of our 14 AVG-pattern features (`YEARS_BEGINEXPLUATATION`,
`FLOORSMAX`) don't appear in this 50-70% list at all — they have lower
missingness (in the 20-50% range from the EDA), so the "41 columns" figure
doesn't map 1:1 onto "all 14 building features." Our redundancy-based
consolidation still applies to all 14 regardless of their exact
missingness percentage — that decision is based on their high correlation
with each other, not on how much of each is missing.

Before dropping two of the three variants per feature (across all 14
building/apartment features, including YEARS_BEGINEXPLUATATION and
FLOORSMAX despite their somewhat lower missingness), we check this
assumption directly — how strongly correlated are the _AVG, _MODE, and
_MEDI versions of each feature with each other? If they're highly
correlated, keeping all three adds redundant information without much
extra predictive value.

Let's start with `COMMONAREA`:

In [11]:
avg = app_train['COMMONAREA_AVG']
mode = app_train['COMMONAREA_MODE']
medi = app_train['COMMONAREA_MEDI']

print('AVG vs MODE:', avg.corr(mode))
print('AVG vs MEDI:', avg.corr(medi))
print('MODE vs MEDI:', mode.corr(medi))

AVG vs MODE: 0.9771470922157294
AVG vs MEDI: 0.9959780551910176
MODE vs MEDI: 0.9798865980420541


The three COMMONAREA variants are extremely highly correlated (0.977-0.996)
— essentially the same underlying information. Keeping all three would add
little value while tripling the number of missing-value columns to handle.

In [12]:
avg2 = app_train['LIVINGAPARTMENTS_AVG']
mode2 = app_train['LIVINGAPARTMENTS_MODE']
medi2 = app_train['LIVINGAPARTMENTS_MEDI']

print('AVG vs MODE:', avg2.corr(mode2))
print('AVG vs MEDI:', avg2.corr(medi2))
print('MODE vs MEDI:', mode2.corr(medi2))

AVG vs MODE: 0.970116678569301
AVG vs MEDI: 0.9938254918064304
MODE vs MEDI: 0.9756053035993285


The same pattern holds for LIVINGAPARTMENTS (0.970-0.994), closely
matching COMMONAREA's correlations. This confirms our assumption
generalizes: across the building/apartment features, the _AVG, _MODE, and
_MEDI variants of the same underlying characteristic are highly redundant.

In [13]:
avg_columns = [col for col in app_train.columns if col.endswith('_AVG')]
print('Number of building/apartment features found:', len(avg_columns))

for avg_col in avg_columns:
    base_name = avg_col.replace('_AVG', '')
    mode_col = base_name + '_MODE'
    medi_col = base_name + '_MEDI'
    
    correlation = app_train[avg_col].corr(app_train[mode_col])
    print(f'{base_name}: AVG vs MODE correlation = {correlation:.3f}')

Number of building/apartment features found: 14
APARTMENTS: AVG vs MODE correlation = 0.973
BASEMENTAREA: AVG vs MODE correlation = 0.973
YEARS_BEGINEXPLUATATION: AVG vs MODE correlation = 0.972
YEARS_BUILD: AVG vs MODE correlation = 0.989
COMMONAREA: AVG vs MODE correlation = 0.977
ELEVATORS: AVG vs MODE correlation = 0.979
ENTRANCES: AVG vs MODE correlation = 0.978
FLOORSMAX: AVG vs MODE correlation = 0.986
FLOORSMIN: AVG vs MODE correlation = 0.986
LANDAREA: AVG vs MODE correlation = 0.974
LIVINGAPARTMENTS: AVG vs MODE correlation = 0.970
LIVINGAREA: AVG vs MODE correlation = 0.972
NONLIVINGAPARTMENTS: AVG vs MODE correlation = 0.969
NONLIVINGAREA: AVG vs MODE correlation = 0.966


All 14 building/apartment features show AVG-MODE correlations between
0.966 and 0.989 — consistently high across the board, not just for the
two examples checked manually. This confirms the assumption generalizes:
we can safely keep only the _AVG variant for each feature and drop the
_MODE and _MEDI versions without losing meaningful information.

Since MODE and MEDI carry essentially the same information as AVG, we
drop them to reduce dimensionality (28 columns removed: 14 features x 2
variants), keeping only the _AVG version of each.

In [14]:
mode_columns = [col for col in app_train.columns 
                if col.endswith('_MODE') and col.replace('_MODE', '_AVG') in app_train.columns]
medi_columns = [col for col in app_train.columns 
                if col.endswith('_MEDI') and col.replace('_MEDI', '_AVG') in app_train.columns]

print('Columns to drop:', len(mode_columns) + len(medi_columns))

app_train = app_train.drop(columns=mode_columns + medi_columns)

print('Shape after dropping:', app_train.shape)

Columns to drop: 28
Shape after dropping: (307511, 220)


Confirmed: 28 columns dropped (14 features x MODE + MEDI), matching expectation. Shape is now 220 — starting from 245, the 3 `EXT_SOURCE_*_MISSING` flag columns added earlier brought us to 248, and dropping the 28 MODE/MEDI columns brings us to 220.

We fill the remaining missing values in the 14 building/apartment _AVG
features with the median. Unlike EXT_SOURCE, we skip separate
missing-indicator flags here, since these features showed only weak
correlations with TARGET in the EDA (e.g. FLOORSMAX_AVG: -0.044) — the
risk of hiding a meaningful signal by imputing is much lower, and adding
14 more flag columns would work against the dimensionality reduction we
just achieved.

In [15]:
building_cols = [col for col in app_train.columns if col.endswith('_AVG')]

print('Missing values before imputation:')
print(app_train[building_cols].isnull().sum().sum())

app_train[building_cols] = app_train[building_cols].fillna(app_train[building_cols].median())

print('Missing values after imputation:')
print(app_train[building_cols].isnull().sum().sum())

Missing values before imputation:
2516080
Missing values after imputation:
0


All missing values across the 14 building/apartment _AVG features are now
filled — 2,516,080 missing values reduced to 0. This total is plausible:
with individual missingness ranging roughly 20-70% across these 14
columns (as seen in the EDA), an average of around 55-60% missing per
column across 307,511 rows and 14 columns lines up closely with this
total. The building/apartment features are now complete and ready for
modeling.

### Handling OWN_CAR_AGE

In our check of the 41 columns with 50-70% missingness, we found one column that is not part of the building/apartment pattern: `OWN_CAR_AGE` (66% missing).

`OWN_CAR_AGE` records the age of an applicant's car. A natural hypothesis: it is missing because the applicant simply does not own a car — not because of a data quality issue. The dataset has a related column, `FLAG_OWN_CAR` (Y/N), which we already one-hot encoded earlier into `FLAG_OWN_CAR_Y` and `FLAG_OWN_CAR_N`.

Before deciding on a treatment, let's verify this hypothesis directly — the same way we cross-checked `DAYS_EMPLOYED` against `OCCUPATION_TYPE` and `ORGANIZATION_TYPE` in the EDA. If the missing `OWN_CAR_AGE` values line up with `FLAG_OWN_CAR_Y == False`, that confirms structural (logical) missingness rather than a data quality gap.

In [16]:
print("Missing OWN_CAR_AGE where FLAG_OWN_CAR_Y is True:", app_train.loc[app_train['FLAG_OWN_CAR_Y'] == True, 'OWN_CAR_AGE'].isnull().sum())
print("Missing OWN_CAR_AGE where FLAG_OWN_CAR_Y is False:", app_train.loc[app_train['FLAG_OWN_CAR_Y'] == False, 'OWN_CAR_AGE'].isnull().sum())
print()
print("Total applicants with FLAG_OWN_CAR_Y True:", (app_train['FLAG_OWN_CAR_Y'] == True).sum())
print("Total applicants with FLAG_OWN_CAR_Y False:", (app_train['FLAG_OWN_CAR_Y'] == False).sum())

Missing OWN_CAR_AGE where FLAG_OWN_CAR_Y is True: 5
Missing OWN_CAR_AGE where FLAG_OWN_CAR_Y is False: 202924

Total applicants with FLAG_OWN_CAR_Y True: 104587
Total applicants with FLAG_OWN_CAR_Y False: 202924


The results confirm the hypothesis almost completely. Among applicants without a car (`FLAG_OWN_CAR_Y == False`, 202,924 total), all 202,924 are missing `OWN_CAR_AGE` — exactly 100%. This confirms the missingness is structural.

There is one small exception worth noting: among applicants who do own a car (`FLAG_OWN_CAR_Y == True`, 104,587 total), 5 also have a missing `OWN_CAR_AGE`. This is a minor data inconsistency rather than part of the structural pattern — only 0.002% of all missing values (5 out of 202,929 total).

Unlike `DAYS_EMPLOYED` and the `EXT_SOURCE` features, we will not create a separate missing-indicator flag here. The reason: a flag column already exists implicitly. During one-hot encoding, `FLAG_OWN_CAR` was converted into `FLAG_OWN_CAR_Y` / `FLAG_OWN_CAR_N`, and `FLAG_OWN_CAR_N` already marks exactly which applicants have no car — the same information a new `OWN_CAR_AGE_MISSING` flag would carry. Adding a second, near-duplicate flag would not give the model any new information.

Instead, we fill the missing values with 0. This is interpretable directly: "no car" becomes a car age of 0, consistent with how the missingness arises. The 5 car owners with missing `OWN_CAR_AGE` will also be filled with 0 — not entirely accurate for these 5 individual cases, but negligible given they represent only 0.0016% of all 307,511 rows.

In [17]:
print("Missing OWN_CAR_AGE before imputation:", app_train['OWN_CAR_AGE'].isnull().sum())

app_train['OWN_CAR_AGE'] = app_train['OWN_CAR_AGE'].fillna(0)

print("Missing OWN_CAR_AGE after imputation:", app_train['OWN_CAR_AGE'].isnull().sum())

Missing OWN_CAR_AGE before imputation: 202929
Missing OWN_CAR_AGE after imputation: 0


Confirmed — the 202,929 missing values in `OWN_CAR_AGE` (matching our earlier count of 202,924 non-car-owners plus 5 car-owner exceptions) have all been replaced with 0. `OWN_CAR_AGE` is now complete, with the missingness fully explained by the existing `FLAG_OWN_CAR_N` column rather than a separate flag.

## Incorporating the Linked Tables

So far, all feature engineering has been done on `application_train` alone — the table describing the current loan application and the applicant's personal and financial situation. Home Credit provides six additional tables that capture the applicant's credit and repayment history from other sources and from earlier Home Credit loans:

- `bureau`: credits reported to a credit bureau, i.e. held at other financial institutions
- `bureau_balance`: monthly balance history for each of those bureau-reported credits
- `previous_application`: the applicant's own earlier applications at Home Credit
- `POS_CASH_balance`, `credit_card_balance`, `installments_payments`: monthly or installment-level history of the applicant's earlier Home Credit loans

This is a natural next step because past repayment behavior is typically one of the strongest predictors of future default risk — often more informative than the application-level features engineered so far. Leaving these tables out would mean ignoring exactly the kind of signal that matters most in real-world credit risk scoring.

There is a structural difference to keep in mind, though: unlike `application_train`, these tables do not have one row per applicant. A single applicant can have several previous credits, and each credit can have several monthly balance entries, so each table can contain multiple rows per `SK_ID_CURR` — the ID column that links every table together. Before any of this information can be merged into `app_train`, each table first needs to be aggregated down to one row per `SK_ID_CURR` (e.g. counts, averages, sums per applicant).

| Table | Contents | One row per | Linked via |
|---|---|---|---|
| `application_train` / `application_test` | Applicant and loan info at the time of the current application; `TARGET` (train only) | applicant | — (the base table) |
| `bureau` | Credits reported to a credit bureau, held at other financial institutions | credit | `SK_ID_CURR` → `application_train`/`test` |
| `bureau_balance` | Monthly balance history for each `bureau` credit | credit-month | `SK_ID_BUREAU` → `bureau` |
| `previous_application` | The applicant's own earlier applications at Home Credit | previous application | `SK_ID_CURR` → `application_train`/`test` |
| `POS_CASH_balance` | Monthly balance history of earlier point-of-sale/cash loans at Home Credit | loan-month | `SK_ID_PREV` → `previous_application`; also carries `SK_ID_CURR` directly |
| `credit_card_balance` | Monthly balance history of earlier credit card loans at Home Credit | loan-month | `SK_ID_PREV` → `previous_application`; also carries `SK_ID_CURR` directly |
| `installments_payments` | Actual payment history for earlier Home Credit loan installments | installment | `SK_ID_PREV` → `previous_application`; also carries `SK_ID_CURR` directly |

### Exploring the `bureau` Table

Before aggregating anything, let's get a first look at the `bureau` table on its own: its shape, its columns, and a few example rows. This gives us the basic facts before we check anything more specific, like whether it truly contains multiple rows per applicant.

In [18]:
bureau = pd.read_csv('/kaggle/input/competitions/home-credit-default-risk/bureau.csv')

print("Shape:", bureau.shape)
print("Columns:", bureau.columns.tolist())
bureau.head()

Shape: (1716428, 17)
Columns: ['SK_ID_CURR', 'SK_ID_BUREAU', 'CREDIT_ACTIVE', 'CREDIT_CURRENCY', 'DAYS_CREDIT', 'CREDIT_DAY_OVERDUE', 'DAYS_CREDIT_ENDDATE', 'DAYS_ENDDATE_FACT', 'AMT_CREDIT_MAX_OVERDUE', 'CNT_CREDIT_PROLONG', 'AMT_CREDIT_SUM', 'AMT_CREDIT_SUM_DEBT', 'AMT_CREDIT_SUM_LIMIT', 'AMT_CREDIT_SUM_OVERDUE', 'CREDIT_TYPE', 'DAYS_CREDIT_UPDATE', 'AMT_ANNUITY']


,SK_ID_CURR,SK_ID_BUREAU,CREDIT_ACTIVE,CREDIT_CURRENCY,DAYS_CREDIT,CREDIT_DAY_OVERDUE,DAYS_CREDIT_ENDDATE,DAYS_ENDDATE_FACT,AMT_CREDIT_MAX_OVERDUE,CNT_CREDIT_PROLONG,AMT_CREDIT_SUM,AMT_CREDIT_SUM_DEBT,AMT_CREDIT_SUM_LIMIT,AMT_CREDIT_SUM_OVERDUE,CREDIT_TYPE,DAYS_CREDIT_UPDATE,AMT_ANNUITY
0,215354,5714462,Closed,currency 1,-497,0,-153.0,-153.0,NaN,0,91323.0,0.0,NaN,0.0,Consumer credit,-131,NaN
1,215354,5714463,Active,currency 1,-208,0,1075.0,NaN,NaN,0,225000.0,171342.0,NaN,0.0,Credit card,-20,NaN
2,215354,5714464,Active,currency 1,-203,0,528.0,NaN,NaN,0,464323.5,NaN,NaN,0.0,Consumer credit,-16,NaN
3,215354,5714465,Active,currency 1,-203,0,NaN,NaN,NaN,0,90000.0,NaN,NaN,0.0,Credit card,-16,NaN
4,215354,5714466,Active,currency 1,-629,0,1197.0,NaN,77674.5,0,2700000.0,NaN,NaN,0.0,Consumer credit,-21,NaN


`bureau` has 1,716,428 rows and 17 columns — already far more rows than `application_train`'s 307,511, which is a first hint that applicants appear multiple times. Each row represents one credit reported to the credit bureau, uniquely identified by `SK_ID_BUREAU`. `SK_ID_CURR` links each credit back to the applicant in `application_train`. The columns describe that credit's status (`CREDIT_ACTIVE`, `CREDIT_TYPE`), its timing relative to the application (`DAYS_CREDIT`, `DAYS_CREDIT_ENDDATE`), and its financial amounts (`AMT_CREDIT_SUM`, `AMT_CREDIT_SUM_DEBT`, `AMT_CREDIT_SUM_OVERDUE`, etc.). Already in the first 5 rows, applicant `215354` appears 5 times with 5 different `SK_ID_BUREAU` values — a first, informal sign of the one-to-many relationship we expected. Let's now confirm this across the entire table rather than relying on this small sample.

In [19]:
print("Total rows in bureau:", bureau.shape[0])
print("Unique applicants (SK_ID_CURR) in bureau:", bureau['SK_ID_CURR'].nunique())
print("Average number of credits per applicant:", bureau.shape[0] / bureau['SK_ID_CURR'].nunique())
print()
print("Total applicants in app_train:", app_train.shape[0])

Total rows in bureau: 1716428
Unique applicants (SK_ID_CURR) in bureau: 305811
Average number of credits per applicant: 5.612708502964249

Total applicants in app_train: 307511


Confirmed: `bureau` does have a genuine one-to-many relationship with `SK_ID_CURR`. Only 305,811 unique applicants account for all 1,716,428 rows, an average of about 5.6 credits per applicant — well above 1, meaning most applicants appear multiple times, exactly as expected.

Comparing this to `application_train`'s 307,511 applicants also reveals something worth carrying forward: 307,511 − 305,811 = 1,700 applicants have no entry in `bureau` at all — no credit reported to the bureau whatsoever. This will matter once we merge the aggregated bureau features back into `app_train`: these 1,700 applicants will end up with missing values for every bureau-derived feature, and we'll need a deliberate decision for how to treat that, similar to the structural-missingness cases we handled earlier (e.g. `OWN_CAR_AGE`).

### Aggregating `bureau` per Applicant

We aggregate `bureau` down to one row per `SK_ID_CURR`, using `groupby()` combined with `.agg()`. Each aggregation is chosen to capture a specific aspect of the applicant's external credit history:

- `BUREAU_CREDIT_COUNT` — the total number of credits on record, a simple measure of how extensive the applicant's credit history is.
- `BUREAU_ACTIVE_COUNT` — how many of those credits are still active, reflecting current outstanding obligations rather than closed, settled ones.
- `BUREAU_CREDIT_SUM_TOTAL` and `BUREAU_CREDIT_SUM_DEBT_TOTAL` — the total credit amount and total remaining debt across all credits, capturing overall financial exposure.
- `BUREAU_CREDIT_SUM_OVERDUE_TOTAL` and `BUREAU_MAX_DAY_OVERDUE` — the total amount currently overdue and the worst overdue duration in days, both direct indicators of repayment behavior.
- `BUREAU_CREDIT_PROLONG_TOTAL` — how often credits were prolonged, which can indicate financial strain.

Counting active credits requires a small preparation step: we create a boolean helper column `IS_ACTIVE` (True where `CREDIT_ACTIVE == 'Active'`) before aggregating, then sum it — the same "flag column first" approach used earlier in the notebook. After aggregating, `.reset_index()` turns `SK_ID_CURR` back into a regular column, which we'll need for merging into `app_train` later.

In [20]:
bureau['IS_ACTIVE'] = bureau['CREDIT_ACTIVE'] == 'Active'

bureau_agg = bureau.groupby('SK_ID_CURR').agg(
    BUREAU_CREDIT_COUNT=('SK_ID_BUREAU', 'count'),
    BUREAU_ACTIVE_COUNT=('IS_ACTIVE', 'sum'),
    BUREAU_CREDIT_SUM_TOTAL=('AMT_CREDIT_SUM', 'sum'),
    BUREAU_CREDIT_SUM_DEBT_TOTAL=('AMT_CREDIT_SUM_DEBT', 'sum'),
    BUREAU_CREDIT_SUM_OVERDUE_TOTAL=('AMT_CREDIT_SUM_OVERDUE', 'sum'),
    BUREAU_MAX_DAY_OVERDUE=('CREDIT_DAY_OVERDUE', 'max'),
    BUREAU_CREDIT_PROLONG_TOTAL=('CNT_CREDIT_PROLONG', 'sum')
).reset_index()

print("Shape of bureau_agg:", bureau_agg.shape)
bureau_agg.head()

Shape of bureau_agg: (305811, 8)


,SK_ID_CURR,BUREAU_CREDIT_COUNT,BUREAU_ACTIVE_COUNT,BUREAU_CREDIT_SUM_TOTAL,BUREAU_CREDIT_SUM_DEBT_TOTAL,BUREAU_CREDIT_SUM_OVERDUE_TOTAL,BUREAU_MAX_DAY_OVERDUE,BUREAU_CREDIT_PROLONG_TOTAL
0,100001,7,3,1453365.000,596686.5,0.0,0,0
1,100002,8,2,865055.565,245781.0,0.0,0,0
2,100003,4,1,1017400.500,0.0,0.0,0,0
3,100004,2,0,189037.800,0.0,0.0,0,0
4,100005,3,2,657126.000,568408.5,0.0,0,0


Confirmed: `bureau_agg` has 305,811 rows — exactly matching the number of unique `SK_ID_CURR` values found earlier — and 8 columns (`SK_ID_CURR` plus the 7 aggregated features). The sample rows look plausible: applicant `100001` has 7 credits on record, 3 still active, and about 597k in outstanding debt against a total credit sum of roughly 1.45M.

Before moving on to the ratio feature, let's run one quick sanity check on the aggregation: the number of active credits can never logically exceed the total number of credits for any applicant. Verifying this directly confirms the aggregation was built correctly, rather than just assuming it.

In [21]:
print("Rows where active count exceeds total count:", (bureau_agg['BUREAU_ACTIVE_COUNT'] > bureau_agg['BUREAU_CREDIT_COUNT']).sum())

Rows where active count exceeds total count: 0


Confirmed: 0 rows where the active credit count exceeds the total credit count. The aggregation is internally consistent.

### Debt-to-Credit Ratio

Beyond the standard aggregates, we add one feature of our own: the ratio of total debt to total credit sum, `BUREAU_DEBT_CREDIT_RATIO = BUREAU_CREDIT_SUM_DEBT_TOTAL / BUREAU_CREDIT_SUM_TOTAL`. This reflects what is commonly called "credit utilization" in credit risk scoring — the fraction of an applicant's total available credit that is currently drawn down as debt. Two applicants can carry the same absolute debt amount and still represent very different risk levels depending on how much of their overall credit capacity that debt represents; the raw sum columns alone don't capture this.

Before computing the ratio, we need to check for a division-by-zero risk: if `BUREAU_CREDIT_SUM_TOTAL` is 0 for some applicants, the ratio would be undefined. Let's check how common that is first, rather than assuming it away.

In [22]:
print("Applicants with BUREAU_CREDIT_SUM_TOTAL == 0:", (bureau_agg['BUREAU_CREDIT_SUM_TOTAL'] == 0).sum())

Applicants with BUREAU_CREDIT_SUM_TOTAL == 0: 1276


Before deciding how to handle these 1,276 cases, we need to check what kind of division-by-zero we're actually dealing with. Unlike plain Python, where dividing by zero raises an error immediately, pandas follows the IEEE-754 floating-point convention: a positive number divided by 0 becomes `inf`, and 0 divided by 0 becomes `NaN` — silently, without any error stopping execution. This is exactly why we check for zero denominators explicitly beforehand: if any of these 1,276 applicants also have debt greater than 0, the ratio would compute to `inf` rather than an undefined 0/0, which would require different handling.

In [23]:
print("Applicants with credit sum 0 AND debt > 0:", ((bureau_agg['BUREAU_CREDIT_SUM_TOTAL'] == 0) & (bureau_agg['BUREAU_CREDIT_SUM_DEBT_TOTAL'] > 0)).sum())

Applicants with credit sum 0 AND debt > 0: 34


The full picture across the 1,276 applicants with zero total credit sum: 1,211 also have zero total debt (0/0, undefined — becomes `NaN`), 34 have positive debt despite zero credit sum (dividing produces `+inf`), and 31 have negative debt despite zero credit sum (dividing produces `-inf`) — both of the latter are data inconsistencies, since debt without any underlying credit sum doesn't logically fit.

We treat all three cases explicitly. The 1,211 no-credit/no-debt applicants get a ratio of 0 — there is nothing to utilize and nothing utilized. The 34 positive-debt cases are capped at 1.0, the natural ceiling for a utilization ratio (fully utilized). The 31 negative-debt cases are floored at 0, since a utilization ratio below 0 has no meaningful interpretation. This avoids introducing `inf` or `-inf` into the dataset, which would otherwise distort any downstream statistic — mean, correlation, scaling — that uses this column.

In [24]:
bureau_agg['BUREAU_DEBT_CREDIT_RATIO'] = bureau_agg['BUREAU_CREDIT_SUM_DEBT_TOTAL'] / bureau_agg['BUREAU_CREDIT_SUM_TOTAL']

print("Positive infinite values before fix:", (bureau_agg['BUREAU_DEBT_CREDIT_RATIO'] == np.inf).sum())
print("Negative infinite values before fix:", (bureau_agg['BUREAU_DEBT_CREDIT_RATIO'] == -np.inf).sum())
print("Missing values before fix:", bureau_agg['BUREAU_DEBT_CREDIT_RATIO'].isnull().sum())

bureau_agg['BUREAU_DEBT_CREDIT_RATIO'] = bureau_agg['BUREAU_DEBT_CREDIT_RATIO'].replace(np.inf, 1.0)
bureau_agg['BUREAU_DEBT_CREDIT_RATIO'] = bureau_agg['BUREAU_DEBT_CREDIT_RATIO'].replace(-np.inf, 0.0)
bureau_agg['BUREAU_DEBT_CREDIT_RATIO'] = bureau_agg['BUREAU_DEBT_CREDIT_RATIO'].fillna(0)

print("Infinite values after fix:", np.isinf(bureau_agg['BUREAU_DEBT_CREDIT_RATIO']).sum())
print("Missing values after fix:", bureau_agg['BUREAU_DEBT_CREDIT_RATIO'].isnull().sum())

Positive infinite values before fix: 34
Negative infinite values before fix: 31
Missing values before fix: 1211
Infinite values after fix: 0
Missing values after fix: 0


### Merging `bureau_agg` into `app_train`

Before merging, we check directly how many `app_train` applicants actually have a matching bureau record, rather than assuming it from the two tables' unique ID counts. `bureau.csv` covers applicants from both `application_train` and `application_test` combined, since bureau records exist independently of which dataset an applicant later belongs to — so `bureau_agg`'s 305,811 unique applicants are not all guaranteed to be part of `application_train`.

In [25]:
matching_ids = app_train['SK_ID_CURR'].isin(bureau_agg['SK_ID_CURR'])

print("app_train applicants WITH a bureau record:", matching_ids.sum())
print("app_train applicants WITHOUT a bureau record:", (~matching_ids).sum())

app_train applicants WITH a bureau record: 263491
app_train applicants WITHOUT a bureau record: 44020


Confirmed: 44,020 applicants in `app_train` have no bureau record — considerably more than a naive comparison of unique ID counts across the two tables would suggest. We now merge `bureau_agg` into `app_train` using a left join on `SK_ID_CURR`, keeping all 307,511 applicants. Since `bureau_agg` has exactly one row per `SK_ID_CURR`, the merge cannot introduce any additional rows, and we expect exactly 44,020 missing values in each new `BUREAU_*` column afterward.

In [26]:
app_train = app_train.merge(bureau_agg, on='SK_ID_CURR', how='left')

print("Shape after merge:", app_train.shape)
print("Missing values in BUREAU_CREDIT_COUNT:", app_train['BUREAU_CREDIT_COUNT'].isnull().sum())

Shape after merge: (307511, 228)
Missing values in BUREAU_CREDIT_COUNT: 44020


Confirmed: 44,020 missing values after the merge, exactly matching the applicants without a bureau record identified above. The merge behaved as expected.

### Filling Missing Bureau Features

The 44,020 applicants without a bureau record have missing values in all 8 new `BUREAU_*` columns. We fill these with 0, following the same logic as `OWN_CAR_AGE`: no bureau record means no credits, no debt, no overdue payments, and nothing to prolong — 0 is the natural, interpretable value for each of these columns, including `BUREAU_DEBT_CREDIT_RATIO` (no credit exposure at all is different from being highly utilized, but both cases are best represented as 0 here, since there's simply no external credit to measure).

In [27]:
bureau_cols = [col for col in app_train.columns if col.startswith('BUREAU_')]
print("Bureau columns to fill:", bureau_cols)

print("Missing values before imputation:")
print(app_train[bureau_cols].isnull().sum().sum())

app_train[bureau_cols] = app_train[bureau_cols].fillna(0)

print("Missing values after imputation:")
print(app_train[bureau_cols].isnull().sum().sum())

Bureau columns to fill: ['BUREAU_CREDIT_COUNT', 'BUREAU_ACTIVE_COUNT', 'BUREAU_CREDIT_SUM_TOTAL', 'BUREAU_CREDIT_SUM_DEBT_TOTAL', 'BUREAU_CREDIT_SUM_OVERDUE_TOTAL', 'BUREAU_MAX_DAY_OVERDUE', 'BUREAU_CREDIT_PROLONG_TOTAL', 'BUREAU_DEBT_CREDIT_RATIO']
Missing values before imputation:
352160
Missing values after imputation:
0


Confirmed: 352,160 missing values (44,020 applicants × 8 columns) filled with 0. All bureau-derived features are now complete, with the "no bureau record" cases entirely explained by the fact that `BUREAU_CREDIT_COUNT == 0` — no separate flag needed, exactly as reasoned above.

### A Deliberate Scope Decision: Leaving Out `bureau_balance`

Of the six linked tables introduced earlier, we deliberately leave out `bureau_balance` — the monthly balance history for each `bureau`-reported credit. Three reasons drive this decision.

First, unlike the other five tables, `bureau_balance` does not link to `SK_ID_CURR` directly; it links via `SK_ID_BUREAU`, meaning it would first need to be aggregated per credit and merged into `bureau` before it could be tied to an applicant at all — an extra step none of the other tables require.

Second, much of the signal `bureau_balance` would add — how consistently an applicant has kept up with monthly payments — already overlaps with information captured in our `bureau` aggregates, such as `CREDIT_DAY_OVERDUE` and `AMT_CREDIT_SUM_OVERDUE`, even if at a coarser, per-credit rather than per-month resolution.

Third, `bureau_balance` is one of the largest tables in the entire dataset, considerably larger than `bureau` itself — the processing effort required is substantial relative to the likely incremental gain on top of what we've already captured.

This is a conscious trade-off for the scope of this project, not an oversight: we prioritize breadth across the remaining tables and targeted, well-reasoned features over exhaustively processing every available table.

### Exploring the `previous_application` Table

We now move to `previous_application` — the applicant's own earlier applications at Home Credit. As with `bureau`, we start with a first look at its shape, columns, and a few example rows before aggregating anything.

In [28]:
previous_application = pd.read_csv('/kaggle/input/competitions/home-credit-default-risk/previous_application.csv')

print("Shape:", previous_application.shape)
print("Columns:", previous_application.columns.tolist())
previous_application.head()

Shape: (1670214, 37)
Columns: ['SK_ID_PREV', 'SK_ID_CURR', 'NAME_CONTRACT_TYPE', 'AMT_ANNUITY', 'AMT_APPLICATION', 'AMT_CREDIT', 'AMT_DOWN_PAYMENT', 'AMT_GOODS_PRICE', 'WEEKDAY_APPR_PROCESS_START', 'HOUR_APPR_PROCESS_START', 'FLAG_LAST_APPL_PER_CONTRACT', 'NFLAG_LAST_APPL_IN_DAY', 'RATE_DOWN_PAYMENT', 'RATE_INTEREST_PRIMARY', 'RATE_INTEREST_PRIVILEGED', 'NAME_CASH_LOAN_PURPOSE', 'NAME_CONTRACT_STATUS', 'DAYS_DECISION', 'NAME_PAYMENT_TYPE', 'CODE_REJECT_REASON', 'NAME_TYPE_SUITE', 'NAME_CLIENT_TYPE', 'NAME_GOODS_CATEGORY', 'NAME_PORTFOLIO', 'NAME_PRODUCT_TYPE', 'CHANNEL_TYPE', 'SELLERPLACE_AREA', 'NAME_SELLER_INDUSTRY', 'CNT_PAYMENT', 'NAME_YIELD_GROUP', 'PRODUCT_COMBINATION', 'DAYS_FIRST_DRAWING', 'DAYS_FIRST_DUE', 'DAYS_LAST_DUE_1ST_VERSION', 'DAYS_LAST_DUE', 'DAYS_TERMINATION', 'NFLAG_INSURED_ON_APPROVAL']


,SK_ID_PREV,SK_ID_CURR,NAME_CONTRACT_TYPE,AMT_ANNUITY,AMT_APPLICATION,AMT_CREDIT,AMT_DOWN_PAYMENT,AMT_GOODS_PRICE,WEEKDAY_APPR_PROCESS_START,HOUR_APPR_PROCESS_START,FLAG_LAST_APPL_PER_CONTRACT,NFLAG_LAST_APPL_IN_DAY,RATE_DOWN_PAYMENT,RATE_INTEREST_PRIMARY,RATE_INTEREST_PRIVILEGED,NAME_CASH_LOAN_PURPOSE,NAME_CONTRACT_STATUS,DAYS_DECISION,NAME_PAYMENT_TYPE,CODE_REJECT_REASON,NAME_TYPE_SUITE,NAME_CLIENT_TYPE,NAME_GOODS_CATEGORY,NAME_PORTFOLIO,NAME_PRODUCT_TYPE,CHANNEL_TYPE,SELLERPLACE_AREA,NAME_SELLER_INDUSTRY,CNT_PAYMENT,NAME_YIELD_GROUP,PRODUCT_COMBINATION,DAYS_FIRST_DRAWING,DAYS_FIRST_DUE,DAYS_LAST_DUE_1ST_VERSION,DAYS_LAST_DUE,DAYS_TERMINATION,NFLAG_INSURED_ON_APPROVAL
0,2030495,271877,Consumer loans,1730.430,17145.0,17145.0,0.0,17145.0,SATURDAY,15,Y,1,0.0,0.182832,0.867336,XAP,Approved,-73,Cash through the bank,XAP,NaN,Repeater,Mobile,POS,XNA,Country-wide,35,Connectivity,12.0,middle,POS mobile with interest,365243.0,-42.0,300.0,-42.0,-37.0,0.0
1,2802425,108129,Cash loans,25188.615,607500.0,679671.0,NaN,607500.0,THURSDAY,11,Y,1,NaN,NaN,NaN,XNA,Approved,-164,XNA,XAP,Unaccompanied,Repeater,XNA,Cash,x-sell,Contact center,-1,XNA,36.0,low_action,Cash X-Sell: low,365243.0,-134.0,916.0,365243.0,365243.0,1.0
2,2523466,122040,Cash loans,15060.735,112500.0,136444.5,NaN,112500.0,TUESDAY,11,Y,1,NaN,NaN,NaN,XNA,Approved,-301,Cash through the bank,XAP,"Spouse, partner",Repeater,XNA,Cash,x-sell,Credit and cash offices,-1,XNA,12.0,high,Cash X-Sell: high,365243.0,-271.0,59.0,365243.0,365243.0,1.0
3,2819243,176158,Cash loans,47041.335,450000.0,470790.0,NaN,450000.0,MONDAY,7,Y,1,NaN,NaN,NaN,XNA,Approved,-512,Cash through the bank,XAP,NaN,Repeater,XNA,Cash,x-sell,Credit and cash offices,-1,XNA,12.0,middle,Cash X-Sell: middle,365243.0,-482.0,-152.0,-182.0,-177.0,1.0
4,1784265,202054,Cash loans,31924.395,337500.0,404055.0,NaN,337500.0,THURSDAY,9,Y,1,NaN,NaN,NaN,Repairs,Refused,-781,Cash through the bank,HC,NaN,Repeater,XNA,Cash,walk-in,Credit and cash offices,-1,XNA,24.0,high,Cash Street: high,NaN,NaN,NaN,NaN,NaN,NaN


`previous_application` has 1,670,214 rows and 37 columns. Each row represents one earlier application at Home Credit, uniquely identified by `SK_ID_PREV`, with `SK_ID_CURR` linking it back to the applicant. Key columns include `NAME_CONTRACT_STATUS` (whether that application was Approved, Refused, Canceled, or an Unused offer), `AMT_APPLICATION` and `AMT_CREDIT` (how much was requested vs. actually granted), and `DAYS_DECISION` (timing relative to the current application).

As with `bureau`, we expect multiple previous applications per applicant. Let's confirm this directly rather than assume it.

In [29]:
print("Total rows in previous_application:", previous_application.shape[0])
print("Unique applicants (SK_ID_CURR):", previous_application['SK_ID_CURR'].nunique())
print("Average number of previous applications per applicant:", previous_application.shape[0] / previous_application['SK_ID_CURR'].nunique())

Total rows in previous_application: 1670214
Unique applicants (SK_ID_CURR): 338857
Average number of previous applications per applicant: 4.928964135313716


Confirmed: `previous_application` also shows a genuine one-to-many relationship — 338,857 unique applicants account for 1,670,214 rows, an average of about 4.9 previous applications per applicant. (Note this unique count is even larger than `bureau`'s, and again larger than `app_train`'s 307,511 — consistent with what we learned earlier: these linked tables span both `application_train` and `application_test`. We'll verify the actual overlap directly when we merge, as before.)

### Aggregating `previous_application` per Applicant

As with `bureau`, we aggregate down to one row per `SK_ID_CURR`. Each aggregation is chosen deliberately:

- `PREV_APP_COUNT` — the total number of previous applications, a measure of how extensive the applicant's history with Home Credit is.
- `PREV_APPROVED_COUNT` and `PREV_REFUSED_COUNT` — how many of those applications were approved versus refused, based on `NAME_CONTRACT_STATUS`.
- `PREV_AMT_CREDIT_MEAN` — the average credit amount actually granted across previous applications.
- `PREV_DAYS_DECISION_MEAN` — the average timing of past decisions, capturing how far back the applicant's history reaches.

Note a deliberate difference from `bureau`: there, we summed amounts, because those columns describe several credits that can all be open and owed *at the same time* — the total exposure matters. Here, each row is a separate, already-concluded application — summing would just reflect how many applications there were (already captured by `PREV_APP_COUNT`) rather than telling us anything new about the typical loan size. We use the mean instead, since it reflects the applicant's typical previous loan size independently of how many applications they made.

Beyond these standard aggregates, we add `PREV_CREDIT_APPLICATION_RATIO` = sum(`AMT_CREDIT`) ÷ sum(`AMT_APPLICATION`) per applicant — the share of the total amount requested across all previous applications that was actually granted. A value noticeably below 1 could indicate that Home Credit has repeatedly approved less than this applicant asked for, a signal the individual amount columns don't directly capture.

In [30]:
previous_application['IS_APPROVED'] = previous_application['NAME_CONTRACT_STATUS'] == 'Approved'
previous_application['IS_REFUSED'] = previous_application['NAME_CONTRACT_STATUS'] == 'Refused'

previous_agg = previous_application.groupby('SK_ID_CURR').agg(
    PREV_APP_COUNT=('SK_ID_PREV', 'count'),
    PREV_APPROVED_COUNT=('IS_APPROVED', 'sum'),
    PREV_REFUSED_COUNT=('IS_REFUSED', 'sum'),
    PREV_AMT_CREDIT_MEAN=('AMT_CREDIT', 'mean'),
    PREV_DAYS_DECISION_MEAN=('DAYS_DECISION', 'mean')
).reset_index()

print("Shape of previous_agg:", previous_agg.shape)
previous_agg.head()

Shape of previous_agg: (338857, 6)


,SK_ID_CURR,PREV_APP_COUNT,PREV_APPROVED_COUNT,PREV_REFUSED_COUNT,PREV_AMT_CREDIT_MEAN,PREV_DAYS_DECISION_MEAN
0,100001,1,1,0,23787.00,-1740.0
1,100002,1,1,0,179055.00,-606.0
2,100003,3,3,0,484191.00,-1305.0
3,100004,1,1,0,20106.00,-815.0
4,100005,2,1,0,20076.75,-536.0


Confirmed: `previous_agg` has 338,857 rows and 6 columns, matching the number of unique applicants found earlier. The sample rows look plausible: applicant `100001` has 1 previous application, approved, with a mean credit amount of about 23,787.

### Credit-to-Application Ratio

We compute `PREV_CREDIT_APPLICATION_RATIO` as sum(`AMT_CREDIT`) ÷ sum(`AMT_APPLICATION`) per applicant, aggregated at the `SK_ID_CURR` level rather than per row — this avoids letting a single unusual previous application dominate the ratio, similar to how we built `BUREAU_DEBT_CREDIT_RATIO`. As before, we check for zero denominators first, since `AMT_APPLICATION` can legitimately be 0 for some loan types in this dataset.

In [31]:
prev_sums = previous_application.groupby('SK_ID_CURR').agg(
    PREV_AMT_CREDIT_SUM=('AMT_CREDIT', 'sum'),
    PREV_AMT_APPLICATION_SUM=('AMT_APPLICATION', 'sum')
).reset_index()

previous_agg = previous_agg.merge(prev_sums, on='SK_ID_CURR', how='left')

print("Applicants with PREV_AMT_APPLICATION_SUM == 0:", (previous_agg['PREV_AMT_APPLICATION_SUM'] == 0).sum())

Applicants with PREV_AMT_APPLICATION_SUM == 0: 1105


Having learned from `bureau` that a zero denominator can hide three different cases (zero/zero, positive/zero, negative/zero), we check all three directly this time rather than discovering them one at a time.

In [32]:
print("Sum 0 and credit > 0:", ((previous_agg['PREV_AMT_APPLICATION_SUM'] == 0) & (previous_agg['PREV_AMT_CREDIT_SUM'] > 0)).sum())
print("Sum 0 and credit < 0:", ((previous_agg['PREV_AMT_APPLICATION_SUM'] == 0) & (previous_agg['PREV_AMT_CREDIT_SUM'] < 0)).sum())
print("Sum 0 and credit == 0:", ((previous_agg['PREV_AMT_APPLICATION_SUM'] == 0) & (previous_agg['PREV_AMT_CREDIT_SUM'] == 0)).sum())

Sum 0 and credit > 0: 853
Sum 0 and credit < 0: 0
Sum 0 and credit == 0: 252


853 + 0 + 252 = 1,105, matching exactly. Unlike `bureau`, there are no negative-value cases here — only zero/zero (252, undefined) and positive-credit/zero-application (853, `+inf`). We treat these the same way as before: the 252 no-application/no-credit applicants get a ratio of 0, and the 853 cases get capped at 1.0, the natural ceiling for this ratio. No flooring at 0 is needed this time, since there are no negative-debt-style cases.

In [33]:
previous_agg['PREV_CREDIT_APPLICATION_RATIO'] = previous_agg['PREV_AMT_CREDIT_SUM'] / previous_agg['PREV_AMT_APPLICATION_SUM']

print("Infinite values before fix:", np.isinf(previous_agg['PREV_CREDIT_APPLICATION_RATIO']).sum())
print("Missing values before fix:", previous_agg['PREV_CREDIT_APPLICATION_RATIO'].isnull().sum())

previous_agg['PREV_CREDIT_APPLICATION_RATIO'] = previous_agg['PREV_CREDIT_APPLICATION_RATIO'].replace(np.inf, 1.0)
previous_agg['PREV_CREDIT_APPLICATION_RATIO'] = previous_agg['PREV_CREDIT_APPLICATION_RATIO'].fillna(0)

print("Infinite values after fix:", np.isinf(previous_agg['PREV_CREDIT_APPLICATION_RATIO']).sum())
print("Missing values after fix:", previous_agg['PREV_CREDIT_APPLICATION_RATIO'].isnull().sum())

Infinite values before fix: 853
Missing values before fix: 252
Infinite values after fix: 0
Missing values after fix: 0


Confirmed: the 853 infinite and 252 missing values were resolved to 1.0 and 0 as expected. `PREV_CREDIT_APPLICATION_RATIO` is now complete.

We drop the two helper columns `PREV_AMT_CREDIT_SUM` and `PREV_AMT_APPLICATION_SUM` now that the ratio is computed — they were only an intermediate step, not part of our planned final feature set.

In [34]:
previous_agg = previous_agg.drop(columns=['PREV_AMT_CREDIT_SUM', 'PREV_AMT_APPLICATION_SUM'])

print("Shape of previous_agg:", previous_agg.shape)
previous_agg.columns.tolist()

Shape of previous_agg: (338857, 7)


['SK_ID_CURR',
 'PREV_APP_COUNT',
 'PREV_APPROVED_COUNT',
 'PREV_REFUSED_COUNT',
 'PREV_AMT_CREDIT_MEAN',
 'PREV_DAYS_DECISION_MEAN',
 'PREV_CREDIT_APPLICATION_RATIO']

### Merging `previous_agg` into `app_train`

As with `bureau`, we check the actual overlap directly before merging, since `previous_application` also spans both `application_train` and `application_test`.

In [35]:
matching_ids = app_train['SK_ID_CURR'].isin(previous_agg['SK_ID_CURR'])

print("app_train applicants WITH previous application history:", matching_ids.sum())
print("app_train applicants WITHOUT previous application history:", (~matching_ids).sum())

app_train applicants WITH previous application history: 291057
app_train applicants WITHOUT previous application history: 16454


Confirmed: 16,454 applicants in `app_train` have no previous application history. We merge `previous_agg` into `app_train` using a left join on `SK_ID_CURR`, expecting exactly 16,454 missing values in each new `PREV_*` column afterward.

In [36]:
app_train = app_train.merge(previous_agg, on='SK_ID_CURR', how='left')

print("Shape after merge:", app_train.shape)
print("Missing values in PREV_APP_COUNT:", app_train['PREV_APP_COUNT'].isnull().sum())

Shape after merge: (307511, 234)
Missing values in PREV_APP_COUNT: 16454


Confirmed: 16,454 missing values after the merge, exactly matching the applicants without previous application history identified above. The merge added the 6 new `PREV_*` columns as expected (228 + 6 = 234).

### Filling Missing Previous Application Features

The 16,454 applicants without previous application history get missing values in all 6 new `PREV_*` columns. Following the same logic as `bureau`, we fill these with 0 — no previous applications means no approvals, no refusals, and nothing to compute a ratio from.

In [37]:
prev_cols = [col for col in app_train.columns if col.startswith('PREV_')]
print("Previous application columns to fill:", prev_cols)

print("Missing values before imputation:")
print(app_train[prev_cols].isnull().sum().sum())

app_train[prev_cols] = app_train[prev_cols].fillna(0)

print("Missing values after imputation:")
print(app_train[prev_cols].isnull().sum().sum())

Previous application columns to fill: ['PREV_APP_COUNT', 'PREV_APPROVED_COUNT', 'PREV_REFUSED_COUNT', 'PREV_AMT_CREDIT_MEAN', 'PREV_DAYS_DECISION_MEAN', 'PREV_CREDIT_APPLICATION_RATIO']
Missing values before imputation:
98724
Missing values after imputation:
0


Confirmed: 98,724 missing values (16,454 applicants × 6 columns) filled with 0. All previous-application-derived features are now complete.

### Exploring the `POS_CASH_balance` Table

We move to `POS_CASH_balance` — the monthly balance history of the applicant's earlier point-of-sale and cash loans at Home Credit. As before, we start with a first look at its shape, columns, and a few example rows.

In [38]:
pos_cash = pd.read_csv('/kaggle/input/competitions/home-credit-default-risk/POS_CASH_balance.csv')

print("Shape:", pos_cash.shape)
print("Columns:", pos_cash.columns.tolist())
pos_cash.head()

Shape: (10001358, 8)
Columns: ['SK_ID_PREV', 'SK_ID_CURR', 'MONTHS_BALANCE', 'CNT_INSTALMENT', 'CNT_INSTALMENT_FUTURE', 'NAME_CONTRACT_STATUS', 'SK_DPD', 'SK_DPD_DEF']


,SK_ID_PREV,SK_ID_CURR,MONTHS_BALANCE,CNT_INSTALMENT,CNT_INSTALMENT_FUTURE,NAME_CONTRACT_STATUS,SK_DPD,SK_DPD_DEF
0,1803195,182943,-31,48.0,45.0,Active,0,0
1,1715348,367990,-33,36.0,35.0,Active,0,0
2,1784872,397406,-32,12.0,9.0,Active,0,0
3,1903291,269225,-35,48.0,42.0,Active,0,0
4,2341044,334279,-35,36.0,35.0,Active,0,0


`POS_CASH_balance` has 10,001,358 rows and 8 columns — by far the largest table we've worked with. Each row represents one monthly snapshot of an earlier point-of-sale or cash loan: `MONTHS_BALANCE` (how many months before the current application), `CNT_INSTALMENT` and `CNT_INSTALMENT_FUTURE` (total and remaining installments), `NAME_CONTRACT_STATUS`, and `SK_DPD` / `SK_DPD_DEF` (days past due, with and without a grace-period tolerance) — a direct measure of payment behavior.

As before, we expect multiple rows per applicant — let's confirm.

In [39]:
print("Total rows in pos_cash:", pos_cash.shape[0])
print("Unique applicants (SK_ID_CURR):", pos_cash['SK_ID_CURR'].nunique())
print("Average number of records per applicant:", pos_cash.shape[0] / pos_cash['SK_ID_CURR'].nunique())

Total rows in pos_cash: 10001358
Unique applicants (SK_ID_CURR): 337252
Average number of records per applicant: 29.655444593360453


### Aggregating `POS_CASH_balance` per Applicant

We aggregate down to one row per `SK_ID_CURR`: `POS_COUNT` (total monthly records, reflecting the length of the payment history), `POS_SK_DPD_MAX` (the worst days-past-due ever recorded), and `POS_SK_DPD_MEAN` (average days past due across the whole history).

In [40]:
pos_cash_agg = pos_cash.groupby('SK_ID_CURR').agg(
    POS_COUNT=('SK_ID_PREV', 'count'),
    POS_SK_DPD_MAX=('SK_DPD', 'max'),
    POS_SK_DPD_MEAN=('SK_DPD', 'mean')
).reset_index()

print("Shape of pos_cash_agg:", pos_cash_agg.shape)
pos_cash_agg.head()

Shape of pos_cash_agg: (337252, 4)


,SK_ID_CURR,POS_COUNT,POS_SK_DPD_MAX,POS_SK_DPD_MEAN
0,100001,9,7,0.777778
1,100002,19,0,0.000000
2,100003,28,0,0.000000
3,100004,4,0,0.000000
4,100005,11,0,0.000000


Confirmed: `pos_cash_agg` has 337,252 rows and 4 columns, matching the unique applicant count. Applicant `100001` shows 9 monthly records with a max delinquency of 7 days.

In [41]:
matching_ids = app_train['SK_ID_CURR'].isin(pos_cash_agg['SK_ID_CURR'])
print("app_train applicants WITH POS_CASH history:", matching_ids.sum())
print("app_train applicants WITHOUT POS_CASH history:", (~matching_ids).sum())

app_train applicants WITH POS_CASH history: 289444
app_train applicants WITHOUT POS_CASH history: 18067


Confirmed: 18,067 applicants in `app_train` have no `POS_CASH_balance` history. We merge with a left join, expecting exactly 18,067 missing values afterward.

In [42]:
app_train = app_train.merge(pos_cash_agg, on='SK_ID_CURR', how='left')

print("Shape after merge:", app_train.shape)
print("Missing values in POS_COUNT:", app_train['POS_COUNT'].isnull().sum())

Shape after merge: (307511, 237)
Missing values in POS_COUNT: 18067


Confirmed: shape and missing count match exactly as expected.

### Filling Missing POS_CASH Features

The 18,067 applicants without `POS_CASH_balance` history get missing values in the 3 new `POS_*` columns. We fill these with 0, consistent with our reasoning so far: no history means no records, no delinquency observed, and nothing to average.

In [43]:
pos_cols = [col for col in app_train.columns if col.startswith('POS_')]
print("POS columns to fill:", pos_cols)

print("Missing values before imputation:")
print(app_train[pos_cols].isnull().sum().sum())

app_train[pos_cols] = app_train[pos_cols].fillna(0)

print("Missing values after imputation:")
print(app_train[pos_cols].isnull().sum().sum())

POS columns to fill: ['POS_COUNT', 'POS_SK_DPD_MAX', 'POS_SK_DPD_MEAN']
Missing values before imputation:
54201
Missing values after imputation:
0


Confirmed: 54,201 missing values (18,067 applicants × 3 columns) filled with 0. `POS_CASH_balance` is fully incorporated.

### Exploring the `credit_card_balance` Table

We move to `credit_card_balance` — the monthly balance history of the applicant's earlier credit card loans at Home Credit. As before, we start with shape, columns, and a few example rows.

In [44]:
credit_card = pd.read_csv('/kaggle/input/competitions/home-credit-default-risk/credit_card_balance.csv')

print("Shape:", credit_card.shape)
print("Columns:", credit_card.columns.tolist())
credit_card.head()

Shape: (3840312, 23)
Columns: ['SK_ID_PREV', 'SK_ID_CURR', 'MONTHS_BALANCE', 'AMT_BALANCE', 'AMT_CREDIT_LIMIT_ACTUAL', 'AMT_DRAWINGS_ATM_CURRENT', 'AMT_DRAWINGS_CURRENT', 'AMT_DRAWINGS_OTHER_CURRENT', 'AMT_DRAWINGS_POS_CURRENT', 'AMT_INST_MIN_REGULARITY', 'AMT_PAYMENT_CURRENT', 'AMT_PAYMENT_TOTAL_CURRENT', 'AMT_RECEIVABLE_PRINCIPAL', 'AMT_RECIVABLE', 'AMT_TOTAL_RECEIVABLE', 'CNT_DRAWINGS_ATM_CURRENT', 'CNT_DRAWINGS_CURRENT', 'CNT_DRAWINGS_OTHER_CURRENT', 'CNT_DRAWINGS_POS_CURRENT', 'CNT_INSTALMENT_MATURE_CUM', 'NAME_CONTRACT_STATUS', 'SK_DPD', 'SK_DPD_DEF']


,SK_ID_PREV,SK_ID_CURR,MONTHS_BALANCE,AMT_BALANCE,AMT_CREDIT_LIMIT_ACTUAL,AMT_DRAWINGS_ATM_CURRENT,AMT_DRAWINGS_CURRENT,AMT_DRAWINGS_OTHER_CURRENT,AMT_DRAWINGS_POS_CURRENT,AMT_INST_MIN_REGULARITY,AMT_PAYMENT_CURRENT,AMT_PAYMENT_TOTAL_CURRENT,AMT_RECEIVABLE_PRINCIPAL,AMT_RECIVABLE,AMT_TOTAL_RECEIVABLE,CNT_DRAWINGS_ATM_CURRENT,CNT_DRAWINGS_CURRENT,CNT_DRAWINGS_OTHER_CURRENT,CNT_DRAWINGS_POS_CURRENT,CNT_INSTALMENT_MATURE_CUM,NAME_CONTRACT_STATUS,SK_DPD,SK_DPD_DEF
0,2562384,378907,-6,56.970,135000,0.0,877.5,0.0,877.5,1700.325,1800.0,1800.0,0.000,0.000,0.000,0.0,1,0.0,1.0,35.0,Active,0,0
1,2582071,363914,-1,63975.555,45000,2250.0,2250.0,0.0,0.0,2250.000,2250.0,2250.0,60175.080,64875.555,64875.555,1.0,1,0.0,0.0,69.0,Active,0,0
2,1740877,371185,-7,31815.225,450000,0.0,0.0,0.0,0.0,2250.000,2250.0,2250.0,26926.425,31460.085,31460.085,0.0,0,0.0,0.0,30.0,Active,0,0
3,1389973,337855,-4,236572.110,225000,2250.0,2250.0,0.0,0.0,11795.760,11925.0,11925.0,224949.285,233048.970,233048.970,1.0,1,0.0,0.0,10.0,Active,0,0
4,1891521,126868,-1,453919.455,450000,0.0,11547.0,0.0,11547.0,22924.890,27000.0,27000.0,443044.395,453919.455,453919.455,0.0,1,0.0,1.0,101.0,Active,0,0


`credit_card_balance` has 3,840,312 rows and 23 columns — each row a monthly snapshot of an earlier Home Credit credit card. Key columns include `AMT_BALANCE` (current balance owed), `AMT_CREDIT_LIMIT_ACTUAL` (the credit limit at that time), various `AMT_DRAWINGS_*` and `CNT_DRAWINGS_*` columns (how much and how often the card was used), `AMT_PAYMENT_CURRENT`/`AMT_PAYMENT_TOTAL_CURRENT` (payments made), and `SK_DPD`/`SK_DPD_DEF` (days past due), the same delinquency measure we used for `POS_CASH_balance`.

### Aggregating `credit_card_balance` per Applicant

We apply the same lightweight approach used for `POS_CASH_balance`: one row per applicant, with a count of records, the average balance, and the maximum delinquency observed. We also carry along the sums of `AMT_BALANCE` and `AMT_CREDIT_LIMIT_ACTUAL` (rather than just their per-row values), since we need these at the aggregate level in the next step to build a credit utilization ratio — computing it from summed totals avoids letting a single anomalous month dominate the result, the same reasoning we used for `BUREAU_DEBT_CREDIT_RATIO`.

In [45]:
credit_card_agg = credit_card.groupby('SK_ID_CURR').agg(
    CC_COUNT=('SK_ID_PREV', 'count'),
    CC_AMT_BALANCE_MEAN=('AMT_BALANCE', 'mean'),
    CC_SK_DPD_MAX=('SK_DPD', 'max'),
    CC_AMT_BALANCE_SUM=('AMT_BALANCE', 'sum'),
    CC_AMT_CREDIT_LIMIT_SUM=('AMT_CREDIT_LIMIT_ACTUAL', 'sum')
).reset_index()

print("Shape of credit_card_agg:", credit_card_agg.shape)
credit_card_agg.head()

Shape of credit_card_agg: (103558, 6)


,SK_ID_CURR,CC_COUNT,CC_AMT_BALANCE_MEAN,CC_SK_DPD_MAX,CC_AMT_BALANCE_SUM,CC_AMT_CREDIT_LIMIT_SUM
0,100006,6,0.000000,0,0.000,1620000
1,100011,74,54482.111149,0,4031676.225,12150000
2,100013,96,18159.919219,1,1743352.245,12645000
3,100021,17,0.000000,0,0.000,11475000
4,100023,8,0.000000,0,0.000,1080000


`credit_card_agg` now holds one row per applicant with 103,558 rows — noticeably fewer than `POS_CASH_balance`'s 337,252, which makes sense since not every applicant with a prior cash loan also held a credit card. The first few rows already show the range we'd expect: some applicants carry a balance of 0 across all their monthly snapshots (e.g. `SK_ID_CURR` 100006, 100021, 100023), while others show substantial average balances (e.g. 54,482 for 100011). `CC_AMT_BALANCE_SUM` and `CC_AMT_CREDIT_LIMIT_SUM` are the two helper columns we'll use next to build `CC_UTILIZATION_RATIO`.

### Building `CC_UTILIZATION_RATIO`

Before dividing `CC_AMT_BALANCE_SUM` by `CC_AMT_CREDIT_LIMIT_SUM`, we again check how many applicants have a credit limit sum of exactly 0 — pandas won't raise a `ZeroDivisionError`, so we have to check this ourselves.

In [46]:
zero_limit = (credit_card_agg['CC_AMT_CREDIT_LIMIT_SUM'] == 0).sum()
print("Applicants with CC_AMT_CREDIT_LIMIT_SUM == 0:", zero_limit)

Applicants with CC_AMT_CREDIT_LIMIT_SUM == 0: 1113


Having learned from `bureau` that a zero denominator can hide three different cases (zero/zero, positive/zero, negative/zero), we check all three directly here as well, rather than assuming which one applies.

In [47]:
zero_and_positive = ((credit_card_agg['CC_AMT_CREDIT_LIMIT_SUM'] == 0) & (credit_card_agg['CC_AMT_BALANCE_SUM'] > 0)).sum()
zero_and_negative = ((credit_card_agg['CC_AMT_CREDIT_LIMIT_SUM'] == 0) & (credit_card_agg['CC_AMT_BALANCE_SUM'] < 0)).sum()
zero_and_zero = ((credit_card_agg['CC_AMT_CREDIT_LIMIT_SUM'] == 0) & (credit_card_agg['CC_AMT_BALANCE_SUM'] == 0)).sum()

print("Sum 0 and balance > 0:", zero_and_positive)
print("Sum 0 and balance < 0:", zero_and_negative)
print("Sum 0 and balance == 0:", zero_and_zero)

Sum 0 and balance > 0: 154
Sum 0 and balance < 0: 0
Sum 0 and balance == 0: 959


The two counts add up exactly: 154 + 959 = 1,113, matching the total number of zero-denominator cases found above. Unlike with `bureau`, there are no cases of a negative balance sum with a zero credit limit sum here, so we only need to handle two of the three possible outcomes: positive divided by zero produces `+inf` (154 cases), and zero divided by zero produces `NaN` (959 cases). We cap the `+inf` cases at 1.0, representing a fully utilized (100%) credit line, and set the `NaN` cases to 0, since an applicant with no balance and no recorded credit limit has nothing to indicate utilization.

In [48]:
credit_card_agg['CC_UTILIZATION_RATIO'] = credit_card_agg['CC_AMT_BALANCE_SUM'] / credit_card_agg['CC_AMT_CREDIT_LIMIT_SUM']

print("Infinite values before fix:", np.isinf(credit_card_agg['CC_UTILIZATION_RATIO']).sum())
print("Missing values before fix:", credit_card_agg['CC_UTILIZATION_RATIO'].isnull().sum())

credit_card_agg['CC_UTILIZATION_RATIO'] = credit_card_agg['CC_UTILIZATION_RATIO'].replace(np.inf, 1.0).replace(-np.inf, 0.0).fillna(0)

print("Infinite values after fix:", np.isinf(credit_card_agg['CC_UTILIZATION_RATIO']).sum())
print("Missing values after fix:", credit_card_agg['CC_UTILIZATION_RATIO'].isnull().sum())

Infinite values before fix: 154
Missing values before fix: 959
Infinite values after fix: 0
Missing values after fix: 0


The fix resolved both cases as expected: the 154 infinite values are now capped at 1.0, and the 959 missing values are now 0. `CC_UTILIZATION_RATIO` is complete for all 103,558 applicants in `credit_card_agg`.

In [49]:
credit_card_agg = credit_card_agg.drop(columns=['CC_AMT_BALANCE_SUM', 'CC_AMT_CREDIT_LIMIT_SUM'])

print("Shape of credit_card_agg:", credit_card_agg.shape)
credit_card_agg.columns.tolist()

Shape of credit_card_agg: (103558, 5)


['SK_ID_CURR',
 'CC_COUNT',
 'CC_AMT_BALANCE_MEAN',
 'CC_SK_DPD_MAX',
 'CC_UTILIZATION_RATIO']

`credit_card_agg` is now ready to merge: `CC_COUNT`, `CC_AMT_BALANCE_MEAN`, `CC_SK_DPD_MAX`, and `CC_UTILIZATION_RATIO`, one row per applicant.

As with the other linked tables, `credit_card_balance` covers both the train and test populations, so we check the overlap with `app_train` directly rather than comparing unique-ID counts.

In [50]:
with_cc = app_train['SK_ID_CURR'].isin(credit_card_agg['SK_ID_CURR']).sum()
without_cc = (~app_train['SK_ID_CURR'].isin(credit_card_agg['SK_ID_CURR'])).sum()

print("app_train applicants WITH credit card history:", with_cc)
print("app_train applicants WITHOUT credit card history:", without_cc)

app_train applicants WITH credit card history: 86905
app_train applicants WITHOUT credit card history: 220606


Only about 28.3% of applicants in `app_train` have a credit card history (86,905 of 307,511). That's noticeably lower than the coverage we saw for `bureau` (85.7%, 263,491 of 307,511) and `previous_application` (94.6%, 291,057 of 307,511) — expected, since a credit card is just one of several loan products, so a much smaller subset of applicants will have used one.

We merge `credit_card_agg` into `app_train` the same way as before, using a left join to keep every applicant.

In [51]:
app_train = app_train.merge(credit_card_agg, on='SK_ID_CURR', how='left')

print("Shape after merge:", app_train.shape)
print("Missing values in CC_COUNT:", app_train['CC_COUNT'].isnull().sum())

Shape after merge: (307511, 241)
Missing values in CC_COUNT: 220606


The merge added the four new `CC_*` columns, and the 220,606 missing values in `CC_COUNT` match exactly the number of applicants without credit card history we found above.

As with `BUREAU_*`, `PREV_*`, and `POS_*`, a missing value here means the applicant has no credit card history — so we fill all `CC_*` columns with 0.

In [52]:
cc_cols = [col for col in app_train.columns if col.startswith('CC_')]

print("Credit card columns to fill:", cc_cols)
print("Missing values before imputation:")
print(app_train[cc_cols].isnull().sum().sum())

app_train[cc_cols] = app_train[cc_cols].fillna(0)

print("Missing values after imputation:")
print(app_train[cc_cols].isnull().sum().sum())

Credit card columns to fill: ['CC_COUNT', 'CC_AMT_BALANCE_MEAN', 'CC_SK_DPD_MAX', 'CC_UTILIZATION_RATIO']
Missing values before imputation:
882424
Missing values after imputation:
0


882,424 missing values (220,606 applicants × 4 columns) are now filled with 0. `credit_card_balance` is fully incorporated into `app_train`.

### Incorporating `installments_payments`

This is the last of the six linked tables. Each row represents one scheduled installment of an earlier Home Credit loan, recording both what was due (`AMT_INSTALMENT`, `DAYS_INSTALMENT`) and what was actually paid (`AMT_PAYMENT`, `DAYS_ENTRY_PAYMENT`). This lets us capture payment behavior directly — whether an applicant tends to pay on time and in full, which is one of the more directly risk-relevant signals among the linked tables. We start the same way as with the other tables: load it and look at its shape and columns.

In [53]:
installments = pd.read_csv('/kaggle/input/competitions/home-credit-default-risk/installments_payments.csv')

print("Shape:", installments.shape)
print("Columns:", installments.columns.tolist())
installments.head()

Shape: (13605401, 8)
Columns: ['SK_ID_PREV', 'SK_ID_CURR', 'NUM_INSTALMENT_VERSION', 'NUM_INSTALMENT_NUMBER', 'DAYS_INSTALMENT', 'DAYS_ENTRY_PAYMENT', 'AMT_INSTALMENT', 'AMT_PAYMENT']


,SK_ID_PREV,SK_ID_CURR,NUM_INSTALMENT_VERSION,NUM_INSTALMENT_NUMBER,DAYS_INSTALMENT,DAYS_ENTRY_PAYMENT,AMT_INSTALMENT,AMT_PAYMENT
0,1054186,161674,1.0,6,-1180.0,-1187.0,6948.360,6948.360
1,1330831,151639,0.0,34,-2156.0,-2156.0,1716.525,1716.525
2,2085231,193053,2.0,1,-63.0,-63.0,25425.000,25425.000
3,2452527,199697,1.0,3,-2418.0,-2426.0,24350.130,24350.130
4,2714724,167756,1.0,2,-1383.0,-1366.0,2165.040,2160.585


`installments_payments` has 13,605,401 rows across 8 columns. The key columns are `AMT_INSTALMENT` (the amount due) versus `AMT_PAYMENT` (the amount actually paid), and `DAYS_INSTALMENT` (when payment was due) versus `DAYS_ENTRY_PAYMENT` (when it was actually made) — together these let us measure both how much of what was owed got paid, and how punctually.

As before, we check how many rows and unique applicants this table covers before aggregating.

In [54]:
print("Total rows in installments:", len(installments))
print("Unique applicants (SK_ID_CURR):", installments['SK_ID_CURR'].nunique())
print("Average number of installments per applicant:", len(installments) / installments['SK_ID_CURR'].nunique())

Total rows in installments: 13605401
Unique applicants (SK_ID_CURR): 339587
Average number of installments per applicant: 40.06455194103425


`installments_payments` covers 339,587 unique applicants — more than any other linked table so far — with an average of about 40 installment records each, reflecting that most loans are paid off over many monthly installments.

Before aggregating, we compute a `DAYS_LATE` column per installment: the difference between when the payment was actually made (`DAYS_ENTRY_PAYMENT`) and when it was due (`DAYS_INSTALMENT`). A positive value means the payment came in late, negative means early. We then aggregate per applicant: a count of installments, the sums of `AMT_PAYMENT` and `AMT_INSTALMENT` (needed for the payment ratio in the next step), and the mean of `DAYS_LATE`.

In [55]:
installments['DAYS_LATE'] = installments['DAYS_ENTRY_PAYMENT'] - installments['DAYS_INSTALMENT']

installments_agg = installments.groupby('SK_ID_CURR').agg(
    INST_COUNT=('SK_ID_PREV', 'count'),
    INST_AMT_PAYMENT_SUM=('AMT_PAYMENT', 'sum'),
    INST_AMT_INSTALMENT_SUM=('AMT_INSTALMENT', 'sum'),
    INST_DAYS_LATE_MEAN=('DAYS_LATE', 'mean')
).reset_index()

print("Shape of installments_agg:", installments_agg.shape)
installments_agg.head()

Shape of installments_agg: (339587, 5)


,SK_ID_CURR,INST_COUNT,INST_AMT_PAYMENT_SUM,INST_AMT_INSTALMENT_SUM,INST_DAYS_LATE_MEAN
0,100001,7,41195.925,41195.925,-7.285714
1,100002,19,219625.695,219625.695,-20.421053
2,100003,25,1618864.650,1618864.650,-7.160000
3,100004,3,21288.465,21288.465,-7.666667
4,100005,9,56161.845,56161.845,-23.555556


The first few rows show `INST_AMT_PAYMENT_SUM` matching `INST_AMT_INSTALMENT_SUM` almost exactly — these applicants paid essentially what was due. `INST_DAYS_LATE_MEAN` is negative for all of them, meaning their payments came in on average before the due date. Whether this pattern holds across the full dataset is something we'll see once we build `INST_PAYMENT_RATIO` next.

Before dividing `INST_AMT_PAYMENT_SUM` by `INST_AMT_INSTALMENT_SUM` to build `INST_PAYMENT_RATIO`, we check for zero denominators as before.

In [56]:
zero_instalment = (installments_agg['INST_AMT_INSTALMENT_SUM'] == 0).sum()
print("Applicants with INST_AMT_INSTALMENT_SUM == 0:", zero_instalment)

Applicants with INST_AMT_INSTALMENT_SUM == 0: 3


As with the other ratio features, we check all three possible zero-denominator cases directly before deciding how to handle them.

In [57]:
zero_and_positive = ((installments_agg['INST_AMT_INSTALMENT_SUM'] == 0) & (installments_agg['INST_AMT_PAYMENT_SUM'] > 0)).sum()
zero_and_negative = ((installments_agg['INST_AMT_INSTALMENT_SUM'] == 0) & (installments_agg['INST_AMT_PAYMENT_SUM'] < 0)).sum()
zero_and_zero = ((installments_agg['INST_AMT_INSTALMENT_SUM'] == 0) & (installments_agg['INST_AMT_PAYMENT_SUM'] == 0)).sum()

print("Sum 0 and payment > 0:", zero_and_positive)
print("Sum 0 and payment < 0:", zero_and_negative)
print("Sum 0 and payment == 0:", zero_and_zero)

Sum 0 and payment > 0: 3
Sum 0 and payment < 0: 0
Sum 0 and payment == 0: 0


All 3 zero-denominator cases fall into the positive/zero case, producing `+inf`. As with the other ratio features, we cap these at 1.0.

In [58]:
installments_agg['INST_PAYMENT_RATIO'] = installments_agg['INST_AMT_PAYMENT_SUM'] / installments_agg['INST_AMT_INSTALMENT_SUM']

print("Infinite values before fix:", np.isinf(installments_agg['INST_PAYMENT_RATIO']).sum())
print("Missing values before fix:", installments_agg['INST_PAYMENT_RATIO'].isnull().sum())

installments_agg['INST_PAYMENT_RATIO'] = installments_agg['INST_PAYMENT_RATIO'].replace(np.inf, 1.0).replace(-np.inf, 0.0).fillna(0)

print("Infinite values after fix:", np.isinf(installments_agg['INST_PAYMENT_RATIO']).sum())
print("Missing values after fix:", installments_agg['INST_PAYMENT_RATIO'].isnull().sum())

Infinite values before fix: 3
Missing values before fix: 0
Infinite values after fix: 0
Missing values after fix: 0


All 3 infinite values are now capped at 1.0. `INST_PAYMENT_RATIO` is complete for all 339,587 applicants.

In [59]:
installments_agg = installments_agg.drop(columns=['INST_AMT_PAYMENT_SUM', 'INST_AMT_INSTALMENT_SUM'])

print("Shape of installments_agg:", installments_agg.shape)
installments_agg.columns.tolist()

Shape of installments_agg: (339587, 4)


['SK_ID_CURR', 'INST_COUNT', 'INST_DAYS_LATE_MEAN', 'INST_PAYMENT_RATIO']

`installments_agg` is ready to merge: `INST_COUNT`, `INST_DAYS_LATE_MEAN`, and `INST_PAYMENT_RATIO`, one row per applicant.

As with the other linked tables, we check the overlap with `app_train` directly before merging.

In [60]:
with_inst = app_train['SK_ID_CURR'].isin(installments_agg['SK_ID_CURR']).sum()
without_inst = (~app_train['SK_ID_CURR'].isin(installments_agg['SK_ID_CURR'])).sum()

print("app_train applicants WITH installment history:", with_inst)
print("app_train applicants WITHOUT installment history:", without_inst)

app_train applicants WITH installment history: 291643
app_train applicants WITHOUT installment history: 15868


About 94.8% of applicants (291,643 of 307,511) have installment payment history — similar coverage to `previous_application`, which makes sense since most previous applications that were approved would generate installment records.

In [61]:
app_train = app_train.merge(installments_agg, on='SK_ID_CURR', how='left')

print("Shape after merge:", app_train.shape)
print("Missing values in INST_COUNT:", app_train['INST_COUNT'].isnull().sum())

Shape after merge: (307511, 244)
Missing values in INST_COUNT: 15868


The merge added the three new `INST_*` columns, and the 15,868 missing values in `INST_COUNT` match exactly the number of applicants without installment history.

As with the other linked tables, a missing value here means no installment history, so we fill all `INST_*` columns with 0. For `INST_DAYS_LATE_MEAN` this isn't a perfect fit — 0 technically means "paid exactly on time" rather than "no history" — but it is the most neutral value we can pick: it doesn't bias these applicants toward either early or late payment behavior.

In [62]:
inst_cols = [col for col in app_train.columns if col.startswith('INST_')]

print("Installment columns to fill:", inst_cols)
print("Missing values before imputation:")
print(app_train[inst_cols].isnull().sum().sum())

app_train[inst_cols] = app_train[inst_cols].fillna(0)

print("Missing values after imputation:")
print(app_train[inst_cols].isnull().sum().sum())

Installment columns to fill: ['INST_COUNT', 'INST_DAYS_LATE_MEAN', 'INST_PAYMENT_RATIO']
Missing values before imputation:
47612
Missing values after imputation:
0


The count of 47,612 missing values doesn't match the naive expectation of 15,868 × 3 = 47,604. We investigate this directly in `installments_agg` before filling the missing values, rather than leaving the mismatch unexplained.

In [63]:
installments_agg[['INST_COUNT', 'INST_DAYS_LATE_MEAN', 'INST_PAYMENT_RATIO']].isnull().sum()

INST_COUNT             0
INST_DAYS_LATE_MEAN    9
INST_PAYMENT_RATIO     0
dtype: int64

In [64]:
nan_days_late_ids = installments_agg[installments_agg['INST_DAYS_LATE_MEAN'].isnull()]['SK_ID_CURR']
print("Of these 9 applicants, how many are in app_train:", app_train['SK_ID_CURR'].isin(nan_days_late_ids).sum())

Of these 9 applicants, how many are in app_train: 8


The count of 47,612 missing values doesn't match the naive expectation of 15,868 × 3 = 47,604. Investigating `installments_agg` directly showed 9 applicants with a missing `INST_DAYS_LATE_MEAN` despite having installment records — these are applicants whose installments are all still unpaid (`DAYS_ENTRY_PAYMENT` is NaN for every one of their rows), so the mean of `DAYS_LATE` has nothing to average and returns NaN. Of these 9, 8 are present in `app_train` (the 9th belongs only to `app_test`, since `installments_agg` spans both populations) — accounting for the extra 8 missing values: 47,604 + 8 = 47,612. Since these 8 applicants are filled with 0 just like the applicants with no installment history at all, the distinction doesn't affect the final data, but it's worth documenting as a genuine edge case rather than leaving the mismatch unexplained.

### Final Missing Value Check

Before summarizing this notebook, we do one final sweep across the entire merged `app_train` dataset to confirm that no missing values remain anywhere — not just in the columns we explicitly imputed, but across all 244 columns.

In [65]:
total_missing = app_train.isnull().sum().sum()
print("Total missing values in app_train:", total_missing)

if total_missing > 0:
    print(app_train.isnull().sum()[app_train.isnull().sum() > 0])

Total missing values in app_train: 457296
AMT_ANNUITY                       12
AMT_GOODS_PRICE                  278
DAYS_EMPLOYED                  55374
CNT_FAM_MEMBERS                    2
TOTALAREA_MODE                148431
OBS_30_CNT_SOCIAL_CIRCLE        1021
DEF_30_CNT_SOCIAL_CIRCLE        1021
OBS_60_CNT_SOCIAL_CIRCLE        1021
DEF_60_CNT_SOCIAL_CIRCLE        1021
DAYS_LAST_PHONE_CHANGE             1
AMT_REQ_CREDIT_BUREAU_HOUR     41519
AMT_REQ_CREDIT_BUREAU_DAY      41519
AMT_REQ_CREDIT_BUREAU_WEEK     41519
AMT_REQ_CREDIT_BUREAU_MON      41519
AMT_REQ_CREDIT_BUREAU_QRT      41519
AMT_REQ_CREDIT_BUREAU_YEAR     41519
dtype: int64


### Handling Remaining Missing Values: Credit Bureau Inquiry Counts

The final sweep revealed six columns we hadn't yet addressed: `AMT_REQ_CREDIT_BUREAU_HOUR/DAY/WEEK/MON/QRT/YEAR`, each recording how many times a credit bureau was checked about the applicant over a given time window (hour, day, week, month, quarter, year). All six show exactly 41,519 missing values — the identical count across all six strongly suggests this isn't a coincidence, but that these are the exact same applicants missing across all six columns simultaneously. Before assuming that and filling with 0, we verify it directly rather than relying on the matching count alone, since two different groups of 41,519 applicants could in principle produce the same total without being the same rows.

In [66]:
req_cols = ['AMT_REQ_CREDIT_BUREAU_HOUR', 'AMT_REQ_CREDIT_BUREAU_DAY', 'AMT_REQ_CREDIT_BUREAU_WEEK',
            'AMT_REQ_CREDIT_BUREAU_MON', 'AMT_REQ_CREDIT_BUREAU_QRT', 'AMT_REQ_CREDIT_BUREAU_YEAR']

missing_mask = app_train[req_cols].isnull()

rows_all_missing = missing_mask.all(axis=1).sum()
rows_any_missing = missing_mask.any(axis=1).sum()

print("Rows missing in ALL 6 columns:", rows_all_missing)
print("Rows missing in AT LEAST ONE column:", rows_any_missing)

Rows missing in ALL 6 columns: 41519
Rows missing in AT LEAST ONE column: 41519


Both counts match exactly at 41,519, confirming this is the same group of applicants missing across all six columns — not two coincidentally equal but different groups. This is consistent with these applicants simply having no credit bureau inquiry data recorded at all, rather than six independent missing-data events.

We fill all six columns with 0, treating a missing inquiry count the same way we treated missing values in the linked tables: no data means no recorded activity, not an unknown value to estimate.

In [67]:
app_train[req_cols] = app_train[req_cols].fillna(0)

print("Missing values before imputation:", missing_mask.sum().sum())
print("Missing values after imputation:", app_train[req_cols].isnull().sum().sum())

Missing values before imputation: 249114
Missing values after imputation: 0


All 249,114 missing values (41,519 applicants × 6 columns) are now filled with 0. The credit bureau inquiry columns are complete.

### Handling Remaining Missing Values: `DAYS_EMPLOYED`

`DAYS_EMPLOYED` still has 55,374 missing values — these are exactly the rows we flagged much earlier with `DAYS_EMPLOYED_ANOM`, where the original value of 365243 (a placeholder for "not currently employed") was replaced with NaN. We created the flag at the time but never actually filled the resulting missing values, so we do that now. Since these NaNs represent a specific, already-understood group (not random missingness) and the flag already captures that information for the model, we impute with the median of the known values.

In [68]:
days_employed_median = app_train['DAYS_EMPLOYED'].median()
print("Median DAYS_EMPLOYED:", days_employed_median)

print("Missing values before imputation:", app_train['DAYS_EMPLOYED'].isnull().sum())
app_train['DAYS_EMPLOYED'] = app_train['DAYS_EMPLOYED'].fillna(days_employed_median)
print("Missing values after imputation:", app_train['DAYS_EMPLOYED'].isnull().sum())

Median DAYS_EMPLOYED: -1648.0
Missing values before imputation: 55374
Missing values after imputation: 0


`DAYS_EMPLOYED` is now complete. The median of -1,648 days (about 4.5 years) was used to fill the 55,374 previously-anomalous values — these applicants are still distinguishable via `DAYS_EMPLOYED_ANOM` for the model, so imputing with a plausible typical value here doesn't lose that information.

### Handling Remaining Missing Values: `TOTALAREA_MODE`

`TOTALAREA_MODE` has 148,431 missing values. Unlike the other `_MODE` columns, it has no matching `_AVG` counterpart, so it survived our earlier drop of redundant MODE/MEDI columns but was never actually imputed. Since it's a building-related feature with a similarly high missingness level to the ones we already handled (weak correlation with `TARGET`, same reasoning as the other building features), we treat it the same way: median imputation, no separate flag.

In [69]:
totalarea_median = app_train['TOTALAREA_MODE'].median()
print("Median TOTALAREA_MODE:", totalarea_median)

print("Missing values before imputation:", app_train['TOTALAREA_MODE'].isnull().sum())
app_train['TOTALAREA_MODE'] = app_train['TOTALAREA_MODE'].fillna(totalarea_median)
print("Missing values after imputation:", app_train['TOTALAREA_MODE'].isnull().sum())

Median TOTALAREA_MODE: 0.0688
Missing values before imputation: 148431
Missing values after imputation: 0


`TOTALAREA_MODE` is now complete, imputed with its median of 0.0688 — consistent with how we treated the other weakly-correlated building features earlier in this notebook.

### Handling Remaining Missing Values: Small Residual Columns

The last eight columns — `AMT_ANNUITY`, `AMT_GOODS_PRICE`, `CNT_FAM_MEMBERS`, `DAYS_LAST_PHONE_CHANGE`, and the four social circle columns (`OBS_30_CNT_SOCIAL_CIRCLE`, `DEF_30_CNT_SOCIAL_CIRCLE`, `OBS_60_CNT_SOCIAL_CIRCLE`, `DEF_60_CNT_SOCIAL_CIRCLE`) — each have very few missing values, ranging from 1 to 1,021 out of 307,511 rows. At this scale, the choice of imputation method has a negligible effect on the overall distribution, so we use median imputation across all of them in a single loop, the same pattern we used earlier for `EXT_SOURCE_1/2/3`.

In [70]:
small_missing_cols = ['AMT_ANNUITY', 'AMT_GOODS_PRICE', 'CNT_FAM_MEMBERS', 'DAYS_LAST_PHONE_CHANGE',
                       'OBS_30_CNT_SOCIAL_CIRCLE', 'DEF_30_CNT_SOCIAL_CIRCLE',
                       'OBS_60_CNT_SOCIAL_CIRCLE', 'DEF_60_CNT_SOCIAL_CIRCLE']

print("Missing values before imputation:")
print(app_train[small_missing_cols].isnull().sum())

for col in small_missing_cols:
    app_train[col] = app_train[col].fillna(app_train[col].median())

print("Missing values after imputation:")
print(app_train[small_missing_cols].isnull().sum())

Missing values before imputation:
AMT_ANNUITY                   12
AMT_GOODS_PRICE              278
CNT_FAM_MEMBERS                2
DAYS_LAST_PHONE_CHANGE         1
OBS_30_CNT_SOCIAL_CIRCLE    1021
DEF_30_CNT_SOCIAL_CIRCLE    1021
OBS_60_CNT_SOCIAL_CIRCLE    1021
DEF_60_CNT_SOCIAL_CIRCLE    1021
dtype: int64
Missing values after imputation:
AMT_ANNUITY                 0
AMT_GOODS_PRICE             0
CNT_FAM_MEMBERS             0
DAYS_LAST_PHONE_CHANGE      0
OBS_30_CNT_SOCIAL_CIRCLE    0
DEF_30_CNT_SOCIAL_CIRCLE    0
OBS_60_CNT_SOCIAL_CIRCLE    0
DEF_60_CNT_SOCIAL_CIRCLE    0
dtype: int64


All eight residual columns are now complete. Combined with the three earlier groups, every missing value uncovered by the final sweep has been addressed.

In [71]:
total_missing = app_train.isnull().sum().sum()
print("Total missing values in app_train:", total_missing)

Total missing values in app_train: 0


`app_train` is now completely free of missing values across all 244 columns, whether from the original application data, our own engineered features, or the six linked tables we incorporated. This closes out the missing value handling for the entire notebook.

## Exporting the Processed Dataset

Kaggle notebooks don't automatically persist a modified dataframe between sessions or notebooks — only files explicitly written to `/kaggle/working/` are saved as "Output" when a version is committed, and only those can be added as "Input" to a different notebook. Since the modeling notebook will need this fully processed `app_train` (all 244 columns, no missing values) without repeating the entire feature engineering process, we export it here.

We use Parquet rather than CSV: CSV doesn't preserve data types, so the boolean columns created by one-hot encoding would be read back as generic objects or strings, requiring extra conversion work. Parquet stores dtypes exactly as they are, and produces a smaller, faster-to-read file for a dataset this size.

In [72]:
app_train.to_parquet('/kaggle/working/app_train_processed.parquet', index=False)

print("Export complete. Shape:", app_train.shape)

Export complete. Shape: (307511, 244)


`app_train_processed.parquet` is now saved to `/kaggle/working/` — once this notebook version is committed, this file becomes available as an Input to the modeling notebook, so the entire feature engineering process doesn't need to be repeated there.